In [ ]:
"""# ================================================================
# M3 scaffold-CV + pseudo-negative augmentation + Applicability Domain
#   - ChEMBL train: consensus_label -> y in {0,1}
#   - Morgan FP (RDKit MorganGenerator): radius=2, nBits=2048
#   - GroupKFold by Murcko scaffold
#   - Pseudo-negatives streamed from COCONUT-A (scaffold-hash split)
#   - Butina clustering for diversity
#   - AD: max Tanimoto similarity to training fold
#   - Report: in-AD vs out-of-AD performance + coverage
#   - Statistics: paired tests + Cohen's dz + bootstrap CI
#
# Outputs:
#   - scaffoldcv_ablation_perfold_with_AD.csv
#   - scaffoldcv_ablation_summary_with_AD.csv
#   - scaffoldcv_AD_diagnostics.csv
# ================================================================

import os
import math
import hashlib
import numpy as np
import pandas as pd

from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import rdFingerprintGenerator
from rdkit.ML.Cluster import Butina

from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    balanced_accuracy_score,
)

RDLogger.DisableLog("rdApp.warning")

from scipy.stats import ttest_rel, wilcoxon

# -------------------------
# Optional: silence RDKit warnings (kekulize etc.)
# -------------------------
# RDLogger.DisableLog("rdApp.warning")

# -------------------------
# PATHS (adjust if needed)
# -------------------------
BASE = r"C:\Users\Besitzer\Desktop\M3_databases"
TRAIN_CSV = os.path.join(BASE, "ChEMBL_M3_consensus_labels_more_negatives_with_meta.csv")
COCO_CSV  = os.path.join(BASE, "coconut_screen_out", "coconut_screen_ranked.csv")
OUTDIR    = os.path.join(BASE, "pseudo_neg_controls_scaffoldcv")
os.makedirs(OUTDIR, exist_ok=True)

# -------------------------
# GLOBAL SETTINGS (publication knobs)
# -------------------------
N_SPLITS = 10                 # <-- increase folds here
RANDOM_STATE = 0              # used for bootstrap only (GroupKFold is deterministic)
AD_THRESHOLD = 0.40           # similarity threshold for "in domain" on test fold
PSEUDO_AD_THRESHOLD = 0.40    # require pseudo-negs to be within AD of training fold (set None to disable)

# Pseudo-neg selection
EPS = 0.01
BUTINA_CUTOFF_DIST = 0.65
MAX_COCO_CANDIDATES = 20000
CHUNKSIZE = 20000

# Fingerprints
MORGAN_RADIUS = 2
MORGAN_NBITS  = 2048
_morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=MORGAN_RADIUS, fpSize=MORGAN_NBITS)

# ================================================================
# (1) Helpers: labels, mol parsing, scaffolds, fingerprints
# ================================================================

def label_to_binary(consensus_label: str):
    if consensus_label in ("active", "active_single"):
        return 1
    if consensus_label in ("inactive", "inactive_single"):
        return 0
    return None

def safe_mol_from_smiles(smiles: str):
    if not isinstance(smiles, str) or not smiles.strip():
        return None
    try:
        return Chem.MolFromSmiles(smiles)
    except Exception:
        return None

def murcko_scaffold_smiles(mol):
    try:
        scaf = MurckoScaffold.GetScaffoldForMol(mol)
        if scaf is None:
            return None
        return Chem.MolToSmiles(scaf, isomericSmiles=False)
    except Exception:
        return None

def mol_to_fp(mol):
    try:
        return _morgan_gen.GetFingerprint(mol)  # ExplicitBitVect-like
    except Exception:
        return None

def fps_to_numpy(fps):
    # scikit-learn expects numeric matrix; ConvertToNumpyArray gives 0/1 array
    X = np.zeros((len(fps), MORGAN_NBITS), dtype=np.uint8)
    for i, fp in enumerate(fps):
        arr = np.zeros((MORGAN_NBITS,), dtype=np.int8)
        DataStructs.ConvertToNumpyArray(fp, arr)
        X[i, :] = arr
    return X

def safe_binary_metrics(y_true, p_pred, threshold=0.5):
    """
    Compute ROC-AUC / PR-AUC / MCC / BalAcc safely.
    Returns dict with NaN for ROC/PR if y_true has only one class.
    """
    y_true = np.asarray(y_true).astype(int)
    p_pred = np.asarray(p_pred).astype(float)

    y_hat = (p_pred >= threshold).astype(int)

    out = {}
    # ROC/PR undefined if only one class present
    if np.unique(y_true).size < 2:
        out["roc_auc"] = np.nan
        out["pr_auc"] = np.nan
    else:
        out["roc_auc"] = float(roc_auc_score(y_true, p_pred))
        out["pr_auc"]  = float(average_precision_score(y_true, p_pred))

    # These are defined even for single-class y_true (but can be trivial)
    out["mcc"]     = float(matthews_corrcoef(y_true, y_hat))
    out["bal_acc"] = float(balanced_accuracy_score(y_true, y_hat))
    return out


def train_lr():
    # Logistic regression baseline used throughout
    return LogisticRegression(
        max_iter=4000,
        solver="lbfgs",
        n_jobs=1
    )

# ================================================================
# (2) Butina clustering (diversity control)
# ================================================================

def butina_cluster_fps(fps, cutoff_dist=0.65):
    if len(fps) == 0:
        return []
    if len(fps) == 1:
        return [[0]]

    dists = []
    for i in range(1, len(fps)):
        sims = DataStructs.BulkTanimotoSimilarity(fps[i], fps[:i])
        dists.extend([1.0 - s for s in sims])

    clusters = Butina.ClusterData(dists, len(fps), cutoff_dist, isDistData=True)
    return [list(c) for c in clusters]

# ================================================================
# (3) Load ChEMBL train, compute scaffolds + fingerprints
# ================================================================

def load_chembl_train(train_csv):
    df = pd.read_csv(train_csv)

    if "consensus_label" not in df.columns:
        raise ValueError("ChEMBL CSV must contain 'consensus_label' column.")

    df["y"] = df["consensus_label"].map(label_to_binary)
    df = df[df["y"].isin([0, 1])].copy()

    smiles_col = None
    for c in ["canonical_smiles", "smiles", "Smiles", "SMILES"]:
        if c in df.columns:
            smiles_col = c
            break
    if smiles_col is None:
        raise ValueError("Could not find a SMILES column in ChEMBL CSV.")

    df["mol"] = df[smiles_col].apply(safe_mol_from_smiles)
    df = df[df["mol"].notnull()].copy()

    df["scaffold"] = df["mol"].apply(murcko_scaffold_smiles)
    df = df[df["scaffold"].notnull()].copy()

    df["fp"] = df["mol"].apply(mol_to_fp)
    df = df[df["fp"].notnull()].copy()

    return df, smiles_col

# ================================================================
# (4) Deterministic scaffold split of COCONUT into A/B by hash
# ================================================================

def scaffold_to_half(scaffold: str, split_ratio=0.5):
    h = hashlib.md5(scaffold.encode("utf-8")).hexdigest()
    x = int(h[:8], 16) / float(16**8)
    return "A" if x < split_ratio else "B"

# ================================================================
# (5) Applicability Domain functions (similarity-based)
# ================================================================

def max_tanimoto_to_train(fp, train_fps):
    # Returns maximum similarity of fp to any fp in train_fps
    sims = DataStructs.BulkTanimotoSimilarity(fp, train_fps)
    return float(max(sims)) if sims else float("nan")

def compute_ad_for_fps(query_fps, train_fps):
    # Vectorized over list (still O(n*m), but n~200-500 per fold is fine)
    return np.array([max_tanimoto_to_train(fp, train_fps) for fp in query_fps], dtype=float)

def eval_metrics(y_true, p_score, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    p_score = np.asarray(p_score).astype(float)
    y_hat = (p_score >= threshold).astype(int)

    out = {}
    if len(y_true) == 0:
        return {"roc_auc": np.nan, "pr_auc": np.nan, "mcc": np.nan, "bal_acc": np.nan,
                "n": 0, "pos": 0, "neg": 0}

    # ROC/PR undefined if only one class present
    if np.unique(y_true).size < 2:
        out["roc_auc"] = np.nan
        out["pr_auc"]  = np.nan
    else:
        out["roc_auc"] = float(roc_auc_score(y_true, p_score))
        out["pr_auc"]  = float(average_precision_score(y_true, p_score))

    out["mcc"] = float(matthews_corrcoef(y_true, y_hat))
    out["bal_acc"] = float(balanced_accuracy_score(y_true, y_hat))

    out["n"] = int(len(y_true))
    out["pos"] = int((y_true == 1).sum())
    out["neg"] = int((y_true == 0).sum())
    return out

# ================================================================
# (6) Stream COCONUT-A, score, and collect pseudo-negative candidates
#     + AD diagnostics for selected pseudo-negs
# ================================================================

def collect_pseudo_neg_candidates(
    coco_csv,
    model,
    eps=0.01,
    max_candidates=20000,
    chunksize=20000,
    smiles_col_guess=("canonical_smiles", "smiles", "SMILES", "Smiles"),
):
    head = pd.read_csv(coco_csv, nrows=5)
    smi_col = None
    for c in smiles_col_guess:
        if c in head.columns:
            smi_col = c
            break
    if smi_col is None:
        raise ValueError("Could not find a SMILES column in COCONUT CSV.")

    kept = []

    for chunk in pd.read_csv(coco_csv, chunksize=chunksize):
        if smi_col not in chunk.columns:
            continue

        smiles_list = chunk[smi_col].astype(str).tolist()

        mols = [safe_mol_from_smiles(s) for s in smiles_list]
        scaffolds = [murcko_scaffold_smiles(m) if m is not None else None for m in mols]

        idx_ok = [i for i, sc in enumerate(scaffolds) if sc is not None]
        if not idx_ok:
            continue

        idx_A = [i for i in idx_ok if scaffold_to_half(scaffolds[i]) == "A"]
        if not idx_A:
            continue

        fps = []
        meta = []
        for i in idx_A:
            fp = mol_to_fp(mols[i])
            if fp is None:
                continue
            fps.append(fp)
            meta.append((smiles_list[i], scaffolds[i]))

        if not fps:
            continue

        X = fps_to_numpy(fps)
        p = model.predict_proba(X)[:, 1]

        for (smi, scaf), pi, fp in zip(meta, p, fps):
            if float(pi) <= eps:
                kept.append({"smiles": smi, "scaffold": scaf, "p": float(pi), "fp": fp})

        if len(kept) > max_candidates:
            kept.sort(key=lambda d: d["p"])
            kept = kept[:max_candidates]

    kept.sort(key=lambda d: d["p"])
    return kept

def select_diverse_pseudo_negs(candidates, n_needed, cutoff_dist=0.65):
    if n_needed <= 0 or len(candidates) == 0:
        return []

    fps = [d["fp"] for d in candidates]
    clusters = butina_cluster_fps(fps, cutoff_dist=cutoff_dist)

    selected = []
    for cl in clusters:
        # pick the "most confidently inactive" (lowest p) representative
        best_idx = min(cl, key=lambda idx: candidates[idx]["p"])
        selected.append(candidates[best_idx])

    selected.sort(key=lambda d: d["p"])
    return selected[:n_needed]

# ================================================================
# (7) One fold runner: baseline or pseudo-neg + AD
# ================================================================

def run_fold_with_ad(
    X_tr, y_tr, fp_tr,
    X_te, y_te, fp_te,
    pseudo_weight=None,
    eps=0.01,
    butina_cutoff_dist=0.65,
    max_coco_candidates=20000,
    ad_threshold=0.40,
    pseudo_ad_threshold=0.40,
):
    """
    Returns:
      - model metrics overall + inAD + outAD
      - AD coverage stats on test fold
      - pseudo-neg diagnostics (counts and sim stats)
    """
    # 1) Train baseline model on training fold
    base_model = train_lr()
    base_model.fit(X_tr, y_tr)

    # 2) Optional pseudo-neg augmentation
    pseudo = []
    pseudo_sim = None

    if pseudo_weight is not None:
        n_pos = int((y_tr == 1).sum())
        n_neg = int((y_tr == 0).sum())
        n_needed = max(0, n_pos - n_neg)

        candidates = collect_pseudo_neg_candidates(
            COCO_CSV,
            model=base_model,
            eps=eps,
            max_candidates=max_coco_candidates,
            chunksize=CHUNKSIZE,
        )

        pseudo = select_diverse_pseudo_negs(
            candidates,
            n_needed=n_needed,
            cutoff_dist=butina_cutoff_dist
        )

        # AD filter for pseudo-negs (recommended for publication)
        if pseudo_ad_threshold is not None and len(pseudo) > 0:
            pseudo_fps = [d["fp"] for d in pseudo]
            pseudo_sim = compute_ad_for_fps(pseudo_fps, fp_tr)
            keep_mask = pseudo_sim >= float(pseudo_ad_threshold)
            pseudo = [d for d, keep in zip(pseudo, keep_mask) if keep]
            pseudo_sim = pseudo_sim[keep_mask] if pseudo_sim is not None else None

        # Build augmented training set
        if len(pseudo) > 0:
            X_pseudo = fps_to_numpy([d["fp"] for d in pseudo])
            y_pseudo = np.zeros((len(pseudo),), dtype=int)

            X_aug = np.vstack([X_tr, X_pseudo])
            y_aug = np.concatenate([y_tr, y_pseudo])

            w = np.ones((len(y_aug),), dtype=float)
            w[len(y_tr):] = float(pseudo_weight)
        else:
            X_aug, y_aug, w = X_tr, y_tr, None

        # Retrain final model
        model = train_lr()
        if w is None:
            model.fit(X_aug, y_aug)
        else:
            model.fit(X_aug, y_aug, sample_weight=w)
    else:
        model = base_model  # baseline run

    # 3) Predict on test fold
    p_te = model.predict_proba(X_te)[:, 1]

    # Full test metrics
    m_all = safe_binary_metrics(y_te, p_te, threshold=0.5)

    # inAD/outAD split
    in_mask  = (ad_scores_te >= ad_threshold)    # or however you define inAD
    out_mask = ~in_mask

    m_in  = safe_binary_metrics(y_te[in_mask],  p_te[in_mask],  threshold=0.5) if in_mask.any()  else None
    m_out = safe_binary_metrics(y_te[out_mask], p_te[out_mask], threshold=0.5) if out_mask.any() else None

    # 4) AD computation on test fold
    te_max_sim = compute_ad_for_fps(fp_te, fp_tr)
    in_ad_mask = te_max_sim >= float(ad_threshold)
    out_ad_mask = ~in_ad_mask

    # 5) Metrics overall + stratified
    overall = eval_metrics(y_te, p_te)

    in_ad = eval_metrics(y_te[in_ad_mask], p_te[in_ad_mask]) if in_ad_mask.any() else {"roc_auc":np.nan,"pr_auc":np.nan,"mcc":np.nan,"bal_acc":np.nan,"n":0,"pos":0,"neg":0}
    out_ad = eval_metrics(y_te[out_ad_mask], p_te[out_ad_mask]) if out_ad_mask.any() else {"roc_auc":np.nan,"pr_auc":np.nan,"mcc":np.nan,"bal_acc":np.nan,"n":0,"pos":0,"neg":0}

    # 6) Coverage + similarity summary
    ad_stats = {
        "ad_threshold": float(ad_threshold),
        "test_in_ad_frac": float(in_ad_mask.mean()),
        "test_in_ad_n": int(in_ad_mask.sum()),
        "test_out_ad_n": int(out_ad_mask.sum()),
        "test_maxsim_mean": float(np.nanmean(te_max_sim)),
        "test_maxsim_median": float(np.nanmedian(te_max_sim)),
        "test_maxsim_p10": float(np.nanpercentile(te_max_sim, 10)),
        "test_maxsim_p90": float(np.nanpercentile(te_max_sim, 90)),
    }

    # 7) Pseudo-neg diagnostics
    pseudo_stats = {
        "n_pseudo": int(len(pseudo)),
        "pseudo_weight": float(pseudo_weight) if pseudo_weight is not None else np.nan,
        "eps": float(eps) if pseudo_weight is not None else np.nan,
        "butina_cutoff_dist": float(butina_cutoff_dist) if pseudo_weight is not None else np.nan,
        "pseudo_ad_threshold": float(pseudo_ad_threshold) if (pseudo_weight is not None and pseudo_ad_threshold is not None) else np.nan,
        "pseudo_maxsim_mean": float(np.nanmean(pseudo_sim)) if (pseudo_sim is not None and len(pseudo_sim)>0) else np.nan,
        "pseudo_maxsim_median": float(np.nanmedian(pseudo_sim)) if (pseudo_sim is not None and len(pseudo_sim)>0) else np.nan,
        "pseudo_maxsim_p10": float(np.nanpercentile(pseudo_sim, 10)) if (pseudo_sim is not None and len(pseudo_sim)>0) else np.nan,
        "pseudo_maxsim_p90": float(np.nanpercentile(pseudo_sim, 90)) if (pseudo_sim is not None and len(pseudo_sim)>0) else np.nan,
    }

    return overall, in_ad, out_ad, ad_stats, pseudo_stats

# ================================================================
# (8) CV runners
# ================================================================

def run_scaffoldcv(chembl_df, model_name, pseudo_weight=None, eps=0.01, butina_cutoff_dist=0.65,
                   max_coco_candidates=20000, n_splits=10, ad_threshold=0.40, pseudo_ad_threshold=0.40):
    """
    model_name: "baseline" or "pseudo_neg"
    pseudo_weight: None for baseline; float for pseudo-neg (e.g., 1.0 or 0.3)
    """
    y = chembl_df["y"].values.astype(int)
    scaff = chembl_df["scaffold"].values
    fp_all = chembl_df["fp"].tolist()
    X = fps_to_numpy(fp_all)

    gkf = GroupKFold(n_splits=n_splits)

    rows = []
    ad_rows = []

    for fold, (tr, te) in enumerate(gkf.split(X, y, groups=scaff), start=1):
        X_tr, y_tr = X[tr], y[tr]
        X_te, y_te = X[te], y[te]
        fp_tr = [fp_all[i] for i in tr]
        fp_te = [fp_all[i] for i in te]

        if pseudo_weight is None:
            print(f"[{model_name} Fold {fold}] running...")
        else:
            n_pos = int((y_tr == 1).sum())
            n_neg = int((y_tr == 0).sum())
            n_need = max(0, n_pos - n_neg)
            print(f"[{model_name} Fold {fold}] train pos={n_pos} neg={n_neg} need pseudo_neg={n_need}")

        overall, in_ad, out_ad, ad_stats, pseudo_stats = run_fold_with_ad(
            X_tr=X_tr, y_tr=y_tr, fp_tr=fp_tr,
            X_te=X_te, y_te=y_te, fp_te=fp_te,
            pseudo_weight=pseudo_weight,
            eps=eps,
            butina_cutoff_dist=butina_cutoff_dist,
            max_coco_candidates=max_coco_candidates,
            ad_threshold=ad_threshold,
            pseudo_ad_threshold=pseudo_ad_threshold,
        )

        # One row per fold with overall + stratified metrics
        row = {
            "model": model_name,
            "fold": fold,
            "n_train_pos": int((y_tr == 1).sum()),
            "n_train_neg": int((y_tr == 0).sum()),
            **pseudo_stats,

            # overall
            "roc_auc": overall["roc_auc"],
            "pr_auc": overall["pr_auc"],
            "mcc": overall["mcc"],
            "bal_acc": overall["bal_acc"],

            # in-AD
            "roc_auc_inAD": in_ad["roc_auc"],
            "pr_auc_inAD": in_ad["pr_auc"],
            "mcc_inAD": in_ad["mcc"],
            "bal_acc_inAD": in_ad["bal_acc"],
            "n_inAD": in_ad["n"],

            # out-AD
            "roc_auc_outAD": out_ad["roc_auc"],
            "pr_auc_outAD": out_ad["pr_auc"],
            "mcc_outAD": out_ad["mcc"],
            "bal_acc_outAD": out_ad["bal_acc"],
            "n_outAD": out_ad["n"],

            # AD stats
            **ad_stats,
        }
        rows.append(row)

        print(f"[{model_name} Fold {fold}] ROC={overall['roc_auc']:.3f} PR={overall['pr_auc']:.3f} MCC={overall['mcc']:.3f} BalAcc={overall['bal_acc']:.3f} | inAD={ad_stats['test_in_ad_frac']:.2f}")

        ad_rows.append({
            "model": model_name,
            "fold": fold,
            **ad_stats,
            **{k: pseudo_stats[k] for k in ["n_pseudo","pseudo_weight","eps","butina_cutoff_dist","pseudo_ad_threshold","pseudo_maxsim_mean","pseudo_maxsim_median","pseudo_maxsim_p10","pseudo_maxsim_p90"]},
            "roc_auc": m_all["roc_auc"],
            "pr_auc":  m_all["pr_auc"],
            "mcc":     m_all["mcc"],
            "bal_acc": m_all["bal_acc"],

            "roc_auc_inAD": m_in["roc_auc"] if m_in else np.nan,
            "pr_auc_inAD":  m_in["pr_auc"]  if m_in else np.nan,
            "mcc_inAD":     m_in["mcc"]     if m_in else np.nan,
            "bal_acc_inAD": m_in["bal_acc"] if m_in else np.nan,

            "roc_auc_outAD": m_out["roc_auc"] if m_out else np.nan,
            "pr_auc_outAD":  m_out["pr_auc"]  if m_out else np.nan,
            "mcc_outAD":     m_out["mcc"]     if m_out else np.nan,
            "bal_acc_outAD": m_out["bal_acc"] if m_out else np.nan
        })

    df = pd.DataFrame(rows)
    df_ad = pd.DataFrame(ad_rows)
    return df, df_ad

def summarize_runs(df_all, metrics):
    group_cols = ["model", "eps", "butina_cutoff_dist", "pseudo_weight", "ad_threshold", "pseudo_ad_threshold"]
    out = (df_all
           .groupby(group_cols, dropna=False)[metrics]
           .agg(["mean", "std"])
           .reset_index())
    out.columns = ["_".join([c for c in col if c]) if isinstance(col, tuple) else col for col in out.columns]
    return out

# ================================================================
# (9) Statistics helpers (paired folds)
# ================================================================

def paired_stats(df_all, model_a, model_b, metric="mcc"):
    """
    Pair by fold between model_a and model_b.
    Returns dict with t-test, Wilcoxon, Cohen's dz, bootstrap CI for mean delta.
    """
    A = df_all[df_all["model"] == model_a][["fold", metric]].rename(columns={metric: f"{metric}_A"})
    B = df_all[df_all["model"] == model_b][["fold", metric]].rename(columns={metric: f"{metric}_B"})
    paired = A.merge(B, on="fold", how="inner").sort_values("fold")

    x = paired[f"{metric}_A"].astype(float).values
    y = paired[f"{metric}_B"].astype(float).values

    # deltas: B - A
    d = y - x
    d = d[np.isfinite(d)]
    n = len(d)

    out = {"metric": metric, "n": int(n)}

    if n < 2:
        out.update({"t_p": np.nan, "t_stat": np.nan, "w_p": np.nan, "w_stat": np.nan, "dz": np.nan,
                    "delta_mean": float(np.nanmean(d)) if n==1 else np.nan,
                    "delta_ci_low": np.nan, "delta_ci_high": np.nan})
        return out

    t = ttest_rel(x, y, nan_policy="omit")
    out["t_stat"] = float(t.statistic)
    out["t_p"] = float(t.pvalue)

    try:
        w = wilcoxon(x, y)
        out["w_stat"] = float(w.statistic)
        out["w_p"] = float(w.pvalue)
    except Exception:
        out["w_stat"] = np.nan
        out["w_p"] = np.nan

    # Cohen's dz (paired): mean(d) / sd(d)
    sd = float(np.std(d, ddof=1))
    out["delta_mean"] = float(np.mean(d))
    out["delta_sd"] = sd
    out["dz"] = float(out["delta_mean"] / sd) if sd > 0 else np.nan

    # Bootstrap CI for mean(d)
    rng = np.random.default_rng(RANDOM_STATE)
    Bn = 5000
    boots = []
    for _ in range(Bn):
        samp = rng.choice(d, size=n, replace=True)
        boots.append(np.mean(samp))
    boots = np.sort(np.array(boots, dtype=float))
    out["delta_ci_low"] = float(np.percentile(boots, 2.5))
    out["delta_ci_high"] = float(np.percentile(boots, 97.5))

    return out

# ================================================================
# (10) Ablation suite + AD
# ================================================================

def run_ablation_suite_with_ad(chembl_df):
    print("\n=== A) Baseline (ChEMBL only) ===")
    df_base, df_base_ad = run_scaffoldcv(
        chembl_df,
        model_name="baseline",
        pseudo_weight=None,
        n_splits=N_SPLITS,
        ad_threshold=AD_THRESHOLD,
        pseudo_ad_threshold=PSEUDO_AD_THRESHOLD,
    )

    print("\n=== B) Pseudo-neg augmentation (weight=1.0) ===")
    df_w1, df_w1_ad = run_scaffoldcv(
        chembl_df,
        model_name="pseudo_w1",
        pseudo_weight=1.0,
        eps=EPS,
        butina_cutoff_dist=BUTINA_CUTOFF_DIST,
        max_coco_candidates=MAX_COCO_CANDIDATES,
        n_splits=N_SPLITS,
        ad_threshold=AD_THRESHOLD,
        pseudo_ad_threshold=PSEUDO_AD_THRESHOLD,
    )

    print("\n=== C) Pseudo-neg augmentation (weight=0.3) ===")
    df_w03, df_w03_ad = run_scaffoldcv(
        chembl_df,
        model_name="pseudo_w03",
        pseudo_weight=0.3,
        eps=EPS,
        butina_cutoff_dist=BUTINA_CUTOFF_DIST,
        max_coco_candidates=MAX_COCO_CANDIDATES,
        n_splits=N_SPLITS,
        ad_threshold=AD_THRESHOLD,
        pseudo_ad_threshold=PSEUDO_AD_THRESHOLD,
    )

    df_all = pd.concat([df_base, df_w1, df_w03], ignore_index=True)
    df_ad  = pd.concat([df_base_ad, df_w1_ad, df_w03_ad], ignore_index=True)

    # Summary includes AD-stratified metrics too
    metrics = [
        "roc_auc","pr_auc","mcc","bal_acc",
        "roc_auc_inAD","pr_auc_inAD","mcc_inAD","bal_acc_inAD",
        "roc_auc_outAD","pr_auc_outAD","mcc_outAD","bal_acc_outAD",
        "test_in_ad_frac","test_maxsim_mean","test_maxsim_median",
        "n_pseudo","pseudo_maxsim_mean","pseudo_maxsim_median"
    ]
    df_sum = summarize_runs(df_all, metrics=metrics)

    return df_all, df_sum, df_ad

# ================================================================
# (11) Main
# ================================================================

def main():
    chembl_df, smiles_col = load_chembl_train(TRAIN_CSV)

    print("Loaded:", chembl_df.shape)
    print("Label counts:\n", chembl_df["consensus_label"].value_counts())

    df_all, df_sum, df_ad = run_ablation_suite_with_ad(chembl_df)

    out_all = os.path.join(OUTDIR, "scaffoldcv_ablation_perfold_with_AD.csv")
    out_sum = os.path.join(OUTDIR, "scaffoldcv_ablation_summary_with_AD.csv")
    out_ad  = os.path.join(OUTDIR, "scaffoldcv_AD_diagnostics.csv")

    df_all.to_csv(out_all, index=False)
    df_sum.to_csv(out_sum, index=False)
    df_ad.to_csv(out_ad, index=False)

    print("\nSaved:")
    print(" -", out_all)
    print(" -", out_sum)
    print(" -", out_ad)

    print("\nAblation summary (mean±sd):")
    print(df_sum)

    # Paired tests: baseline vs pseudo_w1 (overall + inAD)
    print("\n=== Paired fold statistics: baseline vs pseudo_w1 ===")
    for metric in ["mcc", "bal_acc", "roc_auc", "mcc_inAD", "bal_acc_inAD"]:
        st = paired_stats(df_all, model_a="baseline", model_b="pseudo_w1", metric=metric)
        print(f"{metric}: Δmean={st['delta_mean']:+.4f} "
              f"(95% CI {st['delta_ci_low']:+.4f} to {st['delta_ci_high']:+.4f}) | "
              f"t p={st['t_p']:.4g} | Wilcoxon p={st['w_p']:.4g} | dz={st['dz']:.3f}")

    # Basic AD coverage report
    cov = (df_all.groupby("model")["test_in_ad_frac"]
           .agg(["mean","std","min","max"])
           .reset_index())
    print("\nAD coverage (test_in_ad_frac):")
    print(cov)

    return df_all, df_sum, df_ad

# Notebook run:
df_all, df_sum, df_ad = main()
"""

Loaded: (2268, 17)
Label counts:
 consensus_label
active_single      1502
inactive_single     463
active              286
inactive             17
Name: count, dtype: int64

=== A) Baseline (ChEMBL only) ===
[baseline Fold 1] running...
[baseline Fold 1] ROC=0.997 PR=0.999 MCC=0.883 BalAcc=0.925 | inAD=0.97
[baseline Fold 2] running...


c:\Users\Besitzer\anaconda3\envs\openms_env\lib\site-packages\sklearn\metrics\_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
c:\Users\Besitzer\anaconda3\envs\openms_env\lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
c:\Users\Besitzer\anaconda3\envs\openms_env\lib\site-packages\sklearn\metrics\_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


[baseline Fold 2] ROC=0.986 PR=0.997 MCC=0.845 BalAcc=0.923 | inAD=0.96
[baseline Fold 3] running...
[baseline Fold 3] ROC=0.996 PR=0.999 MCC=0.875 BalAcc=0.924 | inAD=0.98
[baseline Fold 4] running...
[baseline Fold 4] ROC=0.981 PR=0.997 MCC=0.766 BalAcc=0.826 | inAD=0.95
[baseline Fold 5] running...
[baseline Fold 5] ROC=0.982 PR=0.996 MCC=0.821 BalAcc=0.925 | inAD=0.93
[baseline Fold 6] running...
[baseline Fold 6] ROC=0.988 PR=0.994 MCC=0.872 BalAcc=0.929 | inAD=0.90
[baseline Fold 7] running...
[baseline Fold 7] ROC=0.983 PR=0.990 MCC=0.851 BalAcc=0.917 | inAD=0.89
[baseline Fold 8] running...
[baseline Fold 8] ROC=0.986 PR=0.997 MCC=0.798 BalAcc=0.931 | inAD=0.96
[baseline Fold 9] running...
[baseline Fold 9] ROC=0.954 PR=0.992 MCC=0.626 BalAcc=0.859 | inAD=0.96
[baseline Fold 10] running...
[baseline Fold 10] ROC=0.972 PR=0.987 MCC=0.705 BalAcc=0.804 | inAD=0.89

=== B) Pseudo-neg augmentation (weight=1.0) ===
[pseudo_w1 Fold 1] train pos=1604 neg=437 need pseudo_neg=1167


[21:40:34] Can't kekulize mol.  Unkekulized atoms: 1 2 3 4 5 6 10 11 15 16 17 19 20 21
[21:40:34] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19
c:\Users\Besitzer\anaconda3\envs\openms_env\lib\site-packages\sklearn\metrics\_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
c:\Users\Besitzer\anaconda3\envs\openms_env\lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
c:\Users\Besitzer\anaconda3\envs\openms_env\lib\site-packages\sklearn\metrics\_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


[pseudo_w1 Fold 1] ROC=0.997 PR=0.999 MCC=0.883 BalAcc=0.925 | inAD=0.97
[pseudo_w1 Fold 2] train pos=1600 neg=441 need pseudo_neg=1159


...

[pseudo_w03 Fold 8] ROC=0.987 PR=0.998 MCC=0.798 BalAcc=0.931 | inAD=0.96
[pseudo_w03 Fold 9] train pos=1597 neg=445 need pseudo_neg=1152


[22:21:21] Can't kekulize mol.  Unkekulized atoms: 1 2 3 4 5 6 10 11 15 16 17 19 20 21
[22:21:21] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19


[pseudo_w03 Fold 9] ROC=0.953 PR=0.991 MCC=0.626 BalAcc=0.859 | inAD=0.96
[pseudo_w03 Fold 10] train pos=1628 neg=414 need pseudo_neg=1214


[22:23:51] Can't kekulize mol.  Unkekulized atoms: 1 2 3 4 5 6 10 11 15 16 17 19 20 21
[22:23:51] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19


[pseudo_w03 Fold 10] ROC=0.971 PR=0.986 MCC=0.739 BalAcc=0.827 | inAD=0.89

Saved:
 - C:\Users\Besitzer\Desktop\M3_databases\pseudo_neg_controls_scaffoldcv\scaffoldcv_ablation_perfold_with_AD.csv
 - C:\Users\Besitzer\Desktop\M3_databases\pseudo_neg_controls_scaffoldcv\scaffoldcv_ablation_summary_with_AD.csv
 - C:\Users\Besitzer\Desktop\M3_databases\pseudo_neg_controls_scaffoldcv\scaffoldcv_AD_diagnostics.csv

Ablation summary (mean±sd):
        model   eps  butina_cutoff_dist  pseudo_weight  ad_threshold  \
0    baseline   NaN                 NaN            NaN           0.4   
1  pseudo_w03  0.01                0.65            0.3           0.4   
2   pseudo_w1  0.01                0.65            1.0           0.4   

   pseudo_ad_threshold  roc_auc_mean  roc_auc_std  pr_auc_mean  pr_auc_std  \
0                  NaN      0.982474     0.012485     0.994847    0.004174   
1                  0.4      0.982437     0.012674     0.994826    0.004238   
2                  0.4      0.982423

Loaded: (2268, 17)
Label counts:
 consensus_label
active_single      1502
inactive_single     463
active              286
inactive             17
Name: count, dtype: int64

=== A) Baseline (ChEMBL only) ===
[baseline Fold 1] running...


c:\Users\Besitzer\anaconda3\envs\openms_env\lib\site-packages\sklearn\metrics\_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
c:\Users\Besitzer\anaconda3\envs\openms_env\lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
c:\Users\Besitzer\anaconda3\envs\openms_env\lib\site-packages\sklearn\metrics\_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


[baseline Fold 1] ROC=0.997 PR=0.999 MCC=0.883 BalAcc=0.925 | inAD=0.97
[baseline Fold 2] running...
[baseline Fold 2] ROC=0.986 PR=0.997 MCC=0.845 BalAcc=0.923 | inAD=0.96
[baseline Fold 3] running...
[baseline Fold 3] ROC=0.996 PR=0.999 MCC=0.875 BalAcc=0.924 | inAD=0.98
[baseline Fold 4] running...
[baseline Fold 4] ROC=0.981 PR=0.997 MCC=0.766 BalAcc=0.826 | inAD=0.95
[baseline Fold 5] running...
[baseline Fold 5] ROC=0.982 PR=0.996 MCC=0.821 BalAcc=0.925 | inAD=0.93
[baseline Fold 6] running...
[baseline Fold 6] ROC=0.988 PR=0.994 MCC=0.872 BalAcc=0.929 | inAD=0.90
[baseline Fold 7] running...
[baseline Fold 7] ROC=0.983 PR=0.990 MCC=0.851 BalAcc=0.917 | inAD=0.89
[baseline Fold 8] running...
[baseline Fold 8] ROC=0.986 PR=0.997 MCC=0.798 BalAcc=0.931 | inAD=0.96
[baseline Fold 9] running...
[baseline Fold 9] ROC=0.954 PR=0.992 MCC=0.626 BalAcc=0.859 | inAD=0.96
[baseline Fold 10] running...
[baseline Fold 10] ROC=0.972 PR=0.987 MCC=0.705 BalAcc=0.804 | inAD=0.89

=== B) Pseudo-ne

[08:16:59] Can't kekulize mol.  Unkekulized atoms: 1 2 3 4 5 6 10 11 15 16 17 19 20 21
[08:16:59] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19
c:\Users\Besitzer\anaconda3\envs\openms_env\lib\site-packages\sklearn\metrics\_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
c:\Users\Besitzer\anaconda3\envs\openms_env\lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
c:\Users\Besitzer\anaconda3\envs\openms_env\lib\site-packages\sklearn\metrics\_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


[pseudo_w1 Fold 1] ROC=0.997 PR=0.999 MCC=0.883 BalAcc=0.925 | inAD=0.97
[pseudo_w1 Fold 2] train pos=1600 neg=441 need pseudo_neg=1159


...

[pseudo_w03 Fold 8] ROC=0.987 PR=0.998 MCC=0.798 BalAcc=0.931 | inAD=0.96
[pseudo_w03 Fold 9] train pos=1597 neg=445 need pseudo_neg=1152


[08:59:19] Can't kekulize mol.  Unkekulized atoms: 1 2 3 4 5 6 10 11 15 16 17 19 20 21
[08:59:19] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19


[pseudo_w03 Fold 9] ROC=0.953 PR=0.991 MCC=0.626 BalAcc=0.859 | inAD=0.96
[pseudo_w03 Fold 10] train pos=1628 neg=414 need pseudo_neg=1214


[09:01:58] Can't kekulize mol.  Unkekulized atoms: 1 2 3 4 5 6 10 11 15 16 17 19 20 21
[09:01:58] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19


[pseudo_w03 Fold 10] ROC=0.971 PR=0.986 MCC=0.739 BalAcc=0.827 | inAD=0.89

Saved:
 - C:\Users\Besitzer\Desktop\M3_databases\pseudo_neg_controls_scaffoldcv\scaffoldcv_ablation_perfold_with_AD.csv
 - C:\Users\Besitzer\Desktop\M3_databases\pseudo_neg_controls_scaffoldcv\scaffoldcv_ablation_summary_with_AD.csv
 - C:\Users\Besitzer\Desktop\M3_databases\pseudo_neg_controls_scaffoldcv\scaffoldcv_AD_diagnostics.csv

Ablation summary (mean±sd):
        model   eps  butina_cutoff_dist  pseudo_weight  ad_threshold  \
0    baseline   NaN                 NaN            NaN           0.4   
1  pseudo_w03  0.01                0.65            0.3           0.4   
2   pseudo_w1  0.01                0.65            1.0           0.4   

   pseudo_ad_threshold  roc_auc_mean  roc_auc_std  pr_auc_mean  pr_auc_std  \
0                  NaN      0.982474     0.012485     0.994847    0.004174   
1                  0.4      0.982437     0.012674     0.994826    0.004238   
2                  0.4      0.982423

In [8]:
# ================================================================
# M3 scaffold-CV + pseudo-negative augmentation + Applicability Domain
# + Publication-ready UMAP plots (global + fold-specific train vs test)
#
#   - ChEMBL train: consensus_label -> y in {0,1}
#   - Morgan FP (RDKit MorganGenerator): radius=2, nBits=2048
#   - GroupKFold by Murcko scaffold
#   - Pseudo-negatives streamed from COCONUT-A (scaffold-hash split)
#   - Butina clustering for diversity
#   - AD: max Tanimoto similarity to training fold
#   - Report: overall + in-AD vs out-of-AD performance + coverage
#   - Statistics: paired tests + Cohen's dz + bootstrap CI
#   - UMAP plots:
#       (A) Global embedding colored by activity
#       (B) Fold-specific embedding: train vs test separation
#           (UMAP fit on train; test transformed)
#
# Outputs:
#   - scaffoldcv_ablation_perfold_with_AD.csv
#   - scaffoldcv_ablation_summary_with_AD.csv
#   - scaffoldcv_AD_diagnostics.csv
#   - figures/UMAP_plots/*.png + *.pdf
#
# Requirements (openms_env):
#   conda install -c conda-forge umap-learn matplotlib
#   (or) pip install umap-learn matplotlib
# ================================================================

import os
import hashlib
import numpy as np
import pandas as pd

from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import rdFingerprintGenerator
from rdkit.ML.Cluster import Butina

from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    balanced_accuracy_score,
)

from scipy.stats import ttest_rel, wilcoxon

# plotting / UMAP
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

try:
    import umap
except Exception as e:
    raise ImportError(
        "UMAP not installed. Install with:\n"
        "  conda install -c conda-forge umap-learn matplotlib\n"
        "or\n"
        "  pip install umap-learn matplotlib"
    ) from e

RDLogger.DisableLog("rdApp.warning")

# -------------------------
# PATHS (adjust if needed)
# -------------------------
BASE = r"C:\Users\Besitzer\Desktop\M3_databases"
TRAIN_CSV = os.path.join(BASE, "ChEMBL_M3_consensus_labels_more_negatives_with_meta.csv")
COCO_CSV  = os.path.join(BASE, "coconut_screen_out", "coconut_screen_ranked.csv")

OUTDIR    = os.path.join(BASE, "pseudo_neg_controls_scaffoldcv")
os.makedirs(OUTDIR, exist_ok=True)

FIGDIR    = os.path.join(BASE, "figures", "UMAP_plots")
os.makedirs(FIGDIR, exist_ok=True)
print("[UMAP] FIGDIR =", FIGDIR)
print("[UMAP] FIGDIR exists:", os.path.isdir(FIGDIR))
_testfile = os.path.join(FIGDIR, "_write_test.txt")
with open(_testfile, "w", encoding="utf-8") as f:
    f.write("ok\n")
print("[UMAP] write test OK:", _testfile)

# -------------------------
# GLOBAL SETTINGS (publication knobs)
# -------------------------
N_SPLITS = 10                 # folds
RANDOM_STATE = 0              # used for bootstrap + UMAP reproducibility
AD_THRESHOLD = 0.40           # similarity threshold for "in domain" on test fold
PSEUDO_AD_THRESHOLD = 0.40    # require pseudo-negs within AD of training fold (None disables)

# Pseudo-neg selection
EPS = 0.01
BUTINA_CUTOFF_DIST = 0.65
MAX_COCO_CANDIDATES = 20000
CHUNKSIZE = 20000

# Fingerprints
MORGAN_RADIUS = 2
MORGAN_NBITS  = 2048
_morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=MORGAN_RADIUS, fpSize=MORGAN_NBITS)

# ================================================================
# (1) Helpers: labels, mol parsing, scaffolds, fingerprints
# ================================================================

def label_to_binary(consensus_label: str):
    if consensus_label in ("active", "active_single"):
        return 1
    if consensus_label in ("inactive", "inactive_single"):
        return 0
    return None

def safe_mol_from_smiles(smiles: str):
    if not isinstance(smiles, str) or not smiles.strip():
        return None
    try:
        return Chem.MolFromSmiles(smiles)
    except Exception:
        return None

def murcko_scaffold_smiles(mol):
    try:
        scaf = MurckoScaffold.GetScaffoldForMol(mol)
        if scaf is None:
            return None
        return Chem.MolToSmiles(scaf, isomericSmiles=False)
    except Exception:
        return None

def mol_to_fp(mol):
    try:
        return _morgan_gen.GetFingerprint(mol)  # ExplicitBitVect-like
    except Exception:
        return None

def fps_to_numpy(fps):
    """Convert list of RDKit bitvectors -> (n, nBits) uint8 matrix."""
    X = np.zeros((len(fps), MORGAN_NBITS), dtype=np.uint8)
    for i, fp in enumerate(fps):
        arr = np.zeros((MORGAN_NBITS,), dtype=np.int8)
        DataStructs.ConvertToNumpyArray(fp, arr)
        X[i, :] = arr
    return X

def safe_binary_metrics(y_true, p_pred, threshold=0.5):
    """
    Compute ROC-AUC / PR-AUC / MCC / BalAcc safely.
    Returns dict with NaN for ROC/PR if y_true has only one class.
    """
    y_true = np.asarray(y_true).astype(int)
    p_pred = np.asarray(p_pred).astype(float)
    y_hat = (p_pred >= threshold).astype(int)

    out = {}
    if np.unique(y_true).size < 2:
        out["roc_auc"] = np.nan
        out["pr_auc"] = np.nan
    else:
        out["roc_auc"] = float(roc_auc_score(y_true, p_pred))
        out["pr_auc"]  = float(average_precision_score(y_true, p_pred))

    out["mcc"]     = float(matthews_corrcoef(y_true, y_hat))
    out["bal_acc"] = float(balanced_accuracy_score(y_true, y_hat))
    out["n"]       = int(len(y_true))
    out["pos"]     = int((y_true == 1).sum())
    out["neg"]     = int((y_true == 0).sum())
    return out

def train_lr():
    return LogisticRegression(
        max_iter=4000,
        solver="lbfgs",
        n_jobs=1
    )

# ================================================================
# (2) Butina clustering (diversity control)
# ================================================================

def butina_cluster_fps(fps, cutoff_dist=0.65):
    if len(fps) == 0:
        return []
    if len(fps) == 1:
        return [[0]]

    dists = []
    for i in range(1, len(fps)):
        sims = DataStructs.BulkTanimotoSimilarity(fps[i], fps[:i])
        dists.extend([1.0 - s for s in sims])

    clusters = Butina.ClusterData(dists, len(fps), cutoff_dist, isDistData=True)
    return [list(c) for c in clusters]

# ================================================================
# (3) Load ChEMBL train, compute scaffolds + fingerprints
# ================================================================

def load_chembl_train(train_csv):
    df = pd.read_csv(train_csv)

    if "consensus_label" not in df.columns:
        raise ValueError("ChEMBL CSV must contain 'consensus_label' column.")

    df["y"] = df["consensus_label"].map(label_to_binary)
    df = df[df["y"].isin([0, 1])].copy()

    smiles_col = None
    for c in ["canonical_smiles", "smiles", "Smiles", "SMILES"]:
        if c in df.columns:
            smiles_col = c
            break
    if smiles_col is None:
        raise ValueError("Could not find a SMILES column in ChEMBL CSV.")

    df["mol"] = df[smiles_col].apply(safe_mol_from_smiles)
    df = df[df["mol"].notnull()].copy()

    df["scaffold"] = df["mol"].apply(murcko_scaffold_smiles)
    df = df[df["scaffold"].notnull()].copy()

    df["fp"] = df["mol"].apply(mol_to_fp)
    df = df[df["fp"].notnull()].copy()

    return df, smiles_col

# ================================================================
# (4) Deterministic scaffold split of COCONUT into A/B by hash
# ================================================================

def scaffold_to_half(scaffold: str, split_ratio=0.5):
    h = hashlib.md5(scaffold.encode("utf-8")).hexdigest()
    x = int(h[:8], 16) / float(16**8)
    return "A" if x < split_ratio else "B"

# ================================================================
# (5) Applicability Domain (similarity-based)
# ================================================================

def max_tanimoto_to_train(fp, train_fps):
    sims = DataStructs.BulkTanimotoSimilarity(fp, train_fps)
    return float(max(sims)) if sims else float("nan")

def compute_ad_for_fps(query_fps, train_fps):
    return np.array([max_tanimoto_to_train(fp, train_fps) for fp in query_fps], dtype=float)

# ================================================================
# (6) Stream COCONUT-A, score, and collect pseudo-negative candidates
#     + AD diagnostics for selected pseudo-negs
# ================================================================

def collect_pseudo_neg_candidates(
    coco_csv,
    model,
    eps=0.01,
    max_candidates=20000,
    chunksize=20000,
    smiles_col_guess=("canonical_smiles", "smiles", "SMILES", "Smiles"),
):
    head = pd.read_csv(coco_csv, nrows=5)
    smi_col = None
    for c in smiles_col_guess:
        if c in head.columns:
            smi_col = c
            break
    if smi_col is None:
        raise ValueError("Could not find a SMILES column in COCONUT CSV.")

    kept = []

    for chunk in pd.read_csv(coco_csv, chunksize=chunksize):
        if smi_col not in chunk.columns:
            continue

        smiles_list = chunk[smi_col].astype(str).tolist()

        mols = [safe_mol_from_smiles(s) for s in smiles_list]
        scaffolds = [murcko_scaffold_smiles(m) if m is not None else None for m in mols]

        idx_ok = [i for i, sc in enumerate(scaffolds) if sc is not None]
        if not idx_ok:
            continue

        idx_A = [i for i in idx_ok if scaffold_to_half(scaffolds[i]) == "A"]
        if not idx_A:
            continue

        fps = []
        meta = []
        for i in idx_A:
            fp = mol_to_fp(mols[i])
            if fp is None:
                continue
            fps.append(fp)
            meta.append((smiles_list[i], scaffolds[i]))

        if not fps:
            continue

        X = fps_to_numpy(fps)
        p = model.predict_proba(X)[:, 1]

        for (smi, scaf), pi, fp in zip(meta, p, fps):
            if float(pi) <= eps:
                kept.append({"smiles": smi, "scaffold": scaf, "p": float(pi), "fp": fp})

        if len(kept) > max_candidates:
            kept.sort(key=lambda d: d["p"])
            kept = kept[:max_candidates]

    kept.sort(key=lambda d: d["p"])
    return kept

def select_diverse_pseudo_negs(candidates, n_needed, cutoff_dist=0.65):
    if n_needed <= 0 or len(candidates) == 0:
        return []

    fps = [d["fp"] for d in candidates]
    clusters = butina_cluster_fps(fps, cutoff_dist=cutoff_dist)

    selected = []
    for cl in clusters:
        best_idx = min(cl, key=lambda idx: candidates[idx]["p"])
        selected.append(candidates[best_idx])

    selected.sort(key=lambda d: d["p"])
    return selected[:n_needed]

# ================================================================
# (7) One fold runner: baseline or pseudo-neg + AD
# ================================================================

def run_fold_with_ad(
    X_tr, y_tr, fp_tr,
    X_te, y_te, fp_te,
    pseudo_weight=None,
    eps=0.01,
    butina_cutoff_dist=0.65,
    max_coco_candidates=20000,
    ad_threshold=0.40,
    pseudo_ad_threshold=0.40,
):
    """
    Returns:
      - overall metrics (dict)
      - inAD metrics (dict)
      - outAD metrics (dict)
      - ad_stats (dict)
      - pseudo_stats (dict)
    """

    # 1) Train baseline model on training fold
    base_model = train_lr()
    base_model.fit(X_tr, y_tr)

    # 2) Optional pseudo-neg augmentation
    pseudo = []
    pseudo_sim = None

    if pseudo_weight is not None:
        n_pos = int((y_tr == 1).sum())
        n_neg = int((y_tr == 0).sum())
        n_needed = max(0, n_pos - n_neg)

        candidates = collect_pseudo_neg_candidates(
            COCO_CSV,
            model=base_model,
            eps=eps,
            max_candidates=max_coco_candidates,
            chunksize=CHUNKSIZE,
        )

        pseudo = select_diverse_pseudo_negs(
            candidates,
            n_needed=n_needed,
            cutoff_dist=butina_cutoff_dist
        )

        # AD filter for pseudo-negs (publication-style hygiene)
        if pseudo_ad_threshold is not None and len(pseudo) > 0:
            pseudo_fps = [d["fp"] for d in pseudo]
            pseudo_sim = compute_ad_for_fps(pseudo_fps, fp_tr)
            keep_mask = pseudo_sim >= float(pseudo_ad_threshold)
            pseudo = [d for d, keep in zip(pseudo, keep_mask) if keep]
            pseudo_sim = pseudo_sim[keep_mask] if pseudo_sim is not None else None

        # Build augmented training set
        if len(pseudo) > 0:
            X_pseudo = fps_to_numpy([d["fp"] for d in pseudo])
            y_pseudo = np.zeros((len(pseudo),), dtype=int)

            X_aug = np.vstack([X_tr, X_pseudo])
            y_aug = np.concatenate([y_tr, y_pseudo])

            w = np.ones((len(y_aug),), dtype=float)
            w[len(y_tr):] = float(pseudo_weight)
        else:
            X_aug, y_aug, w = X_tr, y_tr, None

        # Retrain final model
        model = train_lr()
        if w is None:
            model.fit(X_aug, y_aug)
        else:
            model.fit(X_aug, y_aug, sample_weight=w)
    else:
        model = base_model

    # 3) Predict on test fold
    p_te = model.predict_proba(X_te)[:, 1]

    # 4) AD computation on test fold
    te_max_sim = compute_ad_for_fps(fp_te, fp_tr)
    in_ad_mask  = te_max_sim >= float(ad_threshold)
    out_ad_mask = ~in_ad_mask

    # 5) Metrics overall + stratified
    overall = safe_binary_metrics(y_te, p_te, threshold=0.5)
    in_ad   = safe_binary_metrics(y_te[in_ad_mask],  p_te[in_ad_mask],  threshold=0.5) if in_ad_mask.any()  else {
        "roc_auc":np.nan,"pr_auc":np.nan,"mcc":np.nan,"bal_acc":np.nan,"n":0,"pos":0,"neg":0
    }
    out_ad  = safe_binary_metrics(y_te[out_ad_mask], p_te[out_ad_mask], threshold=0.5) if out_ad_mask.any() else {
        "roc_auc":np.nan,"pr_auc":np.nan,"mcc":np.nan,"bal_acc":np.nan,"n":0,"pos":0,"neg":0
    }

    # 6) Coverage + similarity summary
    ad_stats = {
        "ad_threshold": float(ad_threshold),
        "test_in_ad_frac": float(in_ad_mask.mean()),
        "test_in_ad_n": int(in_ad_mask.sum()),
        "test_out_ad_n": int(out_ad_mask.sum()),
        "test_maxsim_mean": float(np.nanmean(te_max_sim)),
        "test_maxsim_median": float(np.nanmedian(te_max_sim)),
        "test_maxsim_p10": float(np.nanpercentile(te_max_sim, 10)),
        "test_maxsim_p90": float(np.nanpercentile(te_max_sim, 90)),
    }

    # 7) Pseudo-neg diagnostics
    pseudo_stats = {
        "n_pseudo": int(len(pseudo)),
        "pseudo_weight": float(pseudo_weight) if pseudo_weight is not None else np.nan,
        "eps": float(eps) if pseudo_weight is not None else np.nan,
        "butina_cutoff_dist": float(butina_cutoff_dist) if pseudo_weight is not None else np.nan,
        "pseudo_ad_threshold": float(pseudo_ad_threshold) if (pseudo_weight is not None and pseudo_ad_threshold is not None) else np.nan,
        "pseudo_maxsim_mean": float(np.nanmean(pseudo_sim)) if (pseudo_sim is not None and len(pseudo_sim) > 0) else np.nan,
        "pseudo_maxsim_median": float(np.nanmedian(pseudo_sim)) if (pseudo_sim is not None and len(pseudo_sim) > 0) else np.nan,
        "pseudo_maxsim_p10": float(np.nanpercentile(pseudo_sim, 10)) if (pseudo_sim is not None and len(pseudo_sim) > 0) else np.nan,
        "pseudo_maxsim_p90": float(np.nanpercentile(pseudo_sim, 90)) if (pseudo_sim is not None and len(pseudo_sim) > 0) else np.nan,
    }

    return overall, in_ad, out_ad, ad_stats, pseudo_stats

# ================================================================
# (8) CV runner
# ================================================================

def run_scaffoldcv(
    chembl_df,
    model_name,
    pseudo_weight=None,
    eps=0.01,
    butina_cutoff_dist=0.65,
    max_coco_candidates=20000,
    n_splits=10,
    ad_threshold=0.40,
    pseudo_ad_threshold=0.40
):
    """
    model_name: "baseline" or "pseudo_w1" etc.
    pseudo_weight: None for baseline; float for pseudo-neg
    """
    y = chembl_df["y"].values.astype(int)
    scaff = chembl_df["scaffold"].values
    fp_all = chembl_df["fp"].tolist()
    X = fps_to_numpy(fp_all)

    gkf = GroupKFold(n_splits=n_splits)

    rows = []
    ad_rows = []

    for fold, (tr, te) in enumerate(gkf.split(X, y, groups=scaff), start=1):
        X_tr, y_tr = X[tr], y[tr]
        X_te, y_te = X[te], y[te]
        fp_tr = [fp_all[i] for i in tr]
        fp_te = [fp_all[i] for i in te]

        if pseudo_weight is None:
            print(f"[{model_name} Fold {fold}] running...")
        else:
            n_pos = int((y_tr == 1).sum())
            n_neg = int((y_tr == 0).sum())
            n_need = max(0, n_pos - n_neg)
            print(f"[{model_name} Fold {fold}] train pos={n_pos} neg={n_neg} need pseudo_neg={n_need}")

        overall, in_ad, out_ad, ad_stats, pseudo_stats = run_fold_with_ad(
            X_tr=X_tr, y_tr=y_tr, fp_tr=fp_tr,
            X_te=X_te, y_te=y_te, fp_te=fp_te,
            pseudo_weight=pseudo_weight,
            eps=eps,
            butina_cutoff_dist=butina_cutoff_dist,
            max_coco_candidates=max_coco_candidates,
            ad_threshold=ad_threshold,
            pseudo_ad_threshold=pseudo_ad_threshold,
        )

        # One row per fold with overall + stratified metrics
        row = {
            "model": model_name,
            "fold": fold,
            "n_train_pos": int((y_tr == 1).sum()),
            "n_train_neg": int((y_tr == 0).sum()),
            **pseudo_stats,

            # overall
            "roc_auc": overall["roc_auc"],
            "pr_auc": overall["pr_auc"],
            "mcc": overall["mcc"],
            "bal_acc": overall["bal_acc"],

            # in-AD
            "roc_auc_inAD": in_ad["roc_auc"],
            "pr_auc_inAD": in_ad["pr_auc"],
            "mcc_inAD": in_ad["mcc"],
            "bal_acc_inAD": in_ad["bal_acc"],
            "n_inAD": in_ad["n"],

            # out-AD
            "roc_auc_outAD": out_ad["roc_auc"],
            "pr_auc_outAD": out_ad["pr_auc"],
            "mcc_outAD": out_ad["mcc"],
            "bal_acc_outAD": out_ad["bal_acc"],
            "n_outAD": out_ad["n"],

            # AD stats
            **ad_stats,
        }
        rows.append(row)

        print(
            f"[{model_name} Fold {fold}] "
            f"ROC={overall['roc_auc']:.3f} PR={overall['pr_auc']:.3f} "
            f"MCC={overall['mcc']:.3f} BalAcc={overall['bal_acc']:.3f} | "
            f"inAD={ad_stats['test_in_ad_frac']:.2f}"
        )

        # diagnostics table (same info, but explicitly recorded per fold)
        ad_rows.append({
            "model": model_name,
            "fold": fold,
            **pseudo_stats,
            **ad_stats,
            "roc_auc": overall["roc_auc"],
            "pr_auc": overall["pr_auc"],
            "mcc": overall["mcc"],
            "bal_acc": overall["bal_acc"],
            "roc_auc_inAD": in_ad["roc_auc"],
            "pr_auc_inAD": in_ad["pr_auc"],
            "mcc_inAD": in_ad["mcc"],
            "bal_acc_inAD": in_ad["bal_acc"],
            "roc_auc_outAD": out_ad["roc_auc"],
            "pr_auc_outAD": out_ad["pr_auc"],
            "mcc_outAD": out_ad["mcc"],
            "bal_acc_outAD": out_ad["bal_acc"],
            "n_inAD": in_ad["n"],
            "n_outAD": out_ad["n"],
        })

    df = pd.DataFrame(rows)
    df_ad = pd.DataFrame(ad_rows)
    return df, df_ad

def summarize_runs(df_all, metrics):
    group_cols = ["model", "eps", "butina_cutoff_dist", "pseudo_weight", "ad_threshold", "pseudo_ad_threshold"]
    out = (df_all
           .groupby(group_cols, dropna=False)[metrics]
           .agg(["mean", "std"])
           .reset_index())
    out.columns = ["_".join([c for c in col if c]) if isinstance(col, tuple) else col for col in out.columns]
    return out

# ================================================================
# (9) Statistics helpers (paired folds)
# ================================================================

def paired_stats(df_all, model_a, model_b, metric="mcc"):
    """
    Pair by fold between model_a and model_b.
    Returns dict with t-test, Wilcoxon, Cohen's dz, bootstrap CI for mean delta.
    Delta is (B - A).
    """
    A = df_all[df_all["model"] == model_a][["fold", metric]].rename(columns={metric: f"{metric}_A"})
    B = df_all[df_all["model"] == model_b][["fold", metric]].rename(columns={metric: f"{metric}_B"})
    paired = A.merge(B, on="fold", how="inner").sort_values("fold")

    x = paired[f"{metric}_A"].astype(float).values
    y = paired[f"{metric}_B"].astype(float).values

    d = y - x
    d = d[np.isfinite(d)]
    n = len(d)

    out = {"metric": metric, "n": int(n)}

    if n < 2:
        out.update({
            "t_p": np.nan, "t_stat": np.nan, "w_p": np.nan, "w_stat": np.nan, "dz": np.nan,
            "delta_mean": float(np.nanmean(d)) if n == 1 else np.nan,
            "delta_sd": np.nan,
            "delta_ci_low": np.nan, "delta_ci_high": np.nan
        })
        return out

    t = ttest_rel(x, y, nan_policy="omit")
    out["t_stat"] = float(t.statistic)
    out["t_p"] = float(t.pvalue)

    try:
        w = wilcoxon(x, y)
        out["w_stat"] = float(w.statistic)
        out["w_p"] = float(w.pvalue)
    except Exception:
        out["w_stat"] = np.nan
        out["w_p"] = np.nan

    sd = float(np.std(d, ddof=1))
    out["delta_mean"] = float(np.mean(d))
    out["delta_sd"] = sd
    out["dz"] = float(out["delta_mean"] / sd) if sd > 0 else np.nan

    rng = np.random.default_rng(RANDOM_STATE)
    Bn = 5000
    boots = []
    for _ in range(Bn):
        samp = rng.choice(d, size=n, replace=True)
        boots.append(np.mean(samp))
    boots = np.sort(np.array(boots, dtype=float))
    out["delta_ci_low"] = float(np.percentile(boots, 2.5))
    out["delta_ci_high"] = float(np.percentile(boots, 97.5))

    return out

# ================================================================
# (10) UMAP plotting (publication-ready)
# ================================================================

def _save_pub_figure(fig, out_prefix, dpi=300, transparent=False):
    png = out_prefix + ".png"
    pdf = out_prefix + ".pdf"
    fig.savefig(png, dpi=dpi, bbox_inches="tight", transparent=transparent)
    fig.savefig(pdf, bbox_inches="tight", transparent=transparent)
    print("[UMAP] wrote:", png, "| exists:", os.path.exists(png), "| bytes:", os.path.getsize(png) if os.path.exists(png) else "NA")
    print("[UMAP] wrote:", pdf, "| exists:", os.path.exists(pdf), "| bytes:", os.path.getsize(pdf) if os.path.exists(pdf) else "NA")

def plot_umap_global(
    chembl_df,
    out_prefix,
    n_bits=2048,
    n_neighbors=15,
    min_dist=0.10,
    random_state=0,
    point_size=40,          # bigger
    alpha=0.55,             # more transparent
    edge_lw=0.35,           # thin black border
    transparent_bg=False,
):
    """
    Global UMAP embedding of ALL ChEMBL points.
    Fit UMAP on full dataset. Color by y (inactive/active).
    """
    fps = chembl_df["fp"].tolist()
    X = fps_to_numpy(fps).astype(np.float32)
    y = chembl_df["y"].values.astype(int)

    reducer = umap.UMAP(
        n_neighbors=int(n_neighbors),
        min_dist=float(min_dist),
        n_components=2,
        metric="jaccard",
        random_state=int(random_state),
        transform_seed=int(random_state),
    )
    emb = reducer.fit_transform(X)

    # fixed class colors (publication-consistent)
    c_inact = "tab:blue"
    c_act   = "tab:orange"

    fig, ax = plt.subplots(figsize=(7.2, 6.2))

    h0 = ax.scatter(
        emb[y == 0, 0], emb[y == 0, 1],
        s=point_size, alpha=alpha,
        c=c_inact,
        edgecolors="black", linewidths=edge_lw,
        label="Inactive (0)"
    )
    h1 = ax.scatter(
        emb[y == 1, 0], emb[y == 1, 1],
        s=point_size, alpha=alpha,
        c=c_act,
        edgecolors="black", linewidths=edge_lw,
        label="Active (1)"
    )

    ax.set_title("UMAP (Morgan FP) — ChEMBL M3 (global)")
    ax.set_xlabel("UMAP-1")
    ax.set_ylabel("UMAP-2")

    # Force order explicitly
    ax.legend(handles=[h0, h1], frameon=False, loc="best", markerscale=1.2)

    fig.tight_layout()
    _save_pub_figure(fig, out_prefix, dpi=300, transparent=transparent_bg)
    plt.close(fig)

# ================================================================
# (11) Ablation suite + AD
# ================================================================

def run_ablation_suite_with_ad(chembl_df):
    print("\n=== A) Baseline (ChEMBL only) ===")
    df_base, df_base_ad = run_scaffoldcv(
        chembl_df,
        model_name="baseline",
        pseudo_weight=None,
        n_splits=N_SPLITS,
        ad_threshold=AD_THRESHOLD,
        pseudo_ad_threshold=PSEUDO_AD_THRESHOLD,
    )

    print("\n=== B) Pseudo-neg augmentation (weight=1.0) ===")
    df_w1, df_w1_ad = run_scaffoldcv(
        chembl_df,
        model_name="pseudo_w1",
        pseudo_weight=1.0,
        eps=EPS,
        butina_cutoff_dist=BUTINA_CUTOFF_DIST,
        max_coco_candidates=MAX_COCO_CANDIDATES,
        n_splits=N_SPLITS,
        ad_threshold=AD_THRESHOLD,
        pseudo_ad_threshold=PSEUDO_AD_THRESHOLD,
    )

    print("\n=== C) Pseudo-neg augmentation (weight=0.3) ===")
    df_w03, df_w03_ad = run_scaffoldcv(
        chembl_df,
        model_name="pseudo_w03",
        pseudo_weight=0.3,
        eps=EPS,
        butina_cutoff_dist=BUTINA_CUTOFF_DIST,
        max_coco_candidates=MAX_COCO_CANDIDATES,
        n_splits=N_SPLITS,
        ad_threshold=AD_THRESHOLD,
        pseudo_ad_threshold=PSEUDO_AD_THRESHOLD,
    )

    df_all = pd.concat([df_base, df_w1, df_w03], ignore_index=True)
    df_ad  = pd.concat([df_base_ad, df_w1_ad, df_w03_ad], ignore_index=True)

    metrics = [
        "roc_auc","pr_auc","mcc","bal_acc",
        "roc_auc_inAD","pr_auc_inAD","mcc_inAD","bal_acc_inAD",
        "roc_auc_outAD","pr_auc_outAD","mcc_outAD","bal_acc_outAD",
        "test_in_ad_frac","test_maxsim_mean","test_maxsim_median",
        "n_pseudo","pseudo_maxsim_mean","pseudo_maxsim_median"
    ]
    df_sum = summarize_runs(df_all, metrics=metrics)

    return df_all, df_sum, df_ad

# ================================================================
# (12) Main
# ================================================================

def main():
    chembl_df, smiles_col = load_chembl_train(TRAIN_CSV)

    print("Loaded:", chembl_df.shape)
    print("Label counts:\n", chembl_df["consensus_label"].value_counts())

    # ---- UMAP figures first (fast, helps orientation) ----
    plot_umap_global(
        chembl_df,
        out_prefix=os.path.join(FIGDIR, "UMAP_global_activity"),
        n_bits=MORGAN_NBITS,
        n_neighbors=15,
        min_dist=0.10,
        random_state=RANDOM_STATE,
        point_size=18
    )
    print("[UMAP] global done")

    def plot_umap_fold_train_test(
        chembl_df,
        out_prefix,
        fold_id=1,
        n_splits=10,
        n_bits=2048,
        n_neighbors=15,
        min_dist=0.10,
        random_state=0,
        point_size=40,
        alpha_train=0.45,
        alpha_test=0.75,
        edge_lw=0.35,
        transparent_bg=False,
    ):
        """
        Fold-specific UMAP: train vs test separation for a given GroupKFold fold.
        IMPORTANT: UMAP is fit on TRAIN only; TEST is transformed into the embedding.
        """
        y = chembl_df["y"].values.astype(int)
        scaff = chembl_df["scaffold"].values
        fp_all = chembl_df["fp"].tolist()
        X = fps_to_numpy(fp_all).astype(np.float32)

        gkf = GroupKFold(n_splits=int(n_splits))
        splits = list(gkf.split(X, y, groups=scaff))
        if not (1 <= int(fold_id) <= len(splits)):
            raise ValueError(f"fold_id must be in [1, {len(splits)}], got {fold_id}")

        tr, te = splits[int(fold_id) - 1]
        X_tr, y_tr = X[tr], y[tr]
        X_te, y_te = X[te], y[te]

        reducer = umap.UMAP(
            n_neighbors=int(n_neighbors),
            min_dist=float(min_dist),
            n_components=2,
            metric="jaccard",
            random_state=int(random_state),
            transform_seed=int(random_state),
        )

        emb_tr = reducer.fit_transform(X_tr)
        emb_te = reducer.transform(X_te)

        c_inact = "tab:blue"
        c_act   = "tab:orange"

        fig, ax = plt.subplots(figsize=(7.6, 6.2))

        # Train: circles
        ax.scatter(
            emb_tr[y_tr == 0, 0], emb_tr[y_tr == 0, 1],
            s=point_size, alpha=alpha_train, marker="o",
            c=c_inact, edgecolors="black", linewidths=edge_lw
        )
        ax.scatter(
            emb_tr[y_tr == 1, 0], emb_tr[y_tr == 1, 1],
            s=point_size, alpha=alpha_train, marker="o",
            c=c_act, edgecolors="black", linewidths=edge_lw
        )

        # Test: triangles
        ax.scatter(
            emb_te[y_te == 0, 0], emb_te[y_te == 0, 1],
            s=point_size, alpha=alpha_test, marker="^",
            c=c_inact, edgecolors="black", linewidths=edge_lw
        )
        ax.scatter(
            emb_te[y_te == 1, 0], emb_te[y_te == 1, 1],
            s=point_size, alpha=alpha_test, marker="^",
            c=c_act, edgecolors="black", linewidths=edge_lw
        )

        ax.set_title(f"UMAP (fit on train) — Fold {int(fold_id)}: Train vs Test")
        ax.set_xlabel("UMAP-1")
        ax.set_ylabel("UMAP-2")

        # Legend: enforce order AND show correct semantics
        legend_elems = [
            Line2D([0], [0], marker="o", linestyle="None",
                markerfacecolor="white", markeredgecolor="black",
                markersize=9, label="Train"),
            Line2D([0], [0], marker="^", linestyle="None",
                markerfacecolor="white", markeredgecolor="black",
                markersize=9, label="Test"),
            Line2D([0], [0], marker="s", linestyle="None",
                markerfacecolor=c_inact, markeredgecolor="black",
                markersize=9, label="Inactive (0)"),
            Line2D([0], [0], marker="s", linestyle="None",
                markerfacecolor=c_act, markeredgecolor="black",
                markersize=9, label="Active (1)"),
        ]
        ax.legend(handles=legend_elems, frameon=False, loc="best", handletextpad=0.4)

        fig.tight_layout()
        _save_pub_figure(fig, out_prefix, dpi=300, transparent=transparent_bg)
        plt.close(fig)

    print("[UMAP] fold plot done")
    
    # ---- CV + AD suite ----
    df_all, df_sum, df_ad = run_ablation_suite_with_ad(chembl_df)

    out_all = os.path.join(OUTDIR, "scaffoldcv_ablation_perfold_with_AD.csv")
    out_sum = os.path.join(OUTDIR, "scaffoldcv_ablation_summary_with_AD.csv")
    out_ad  = os.path.join(OUTDIR, "scaffoldcv_AD_diagnostics.csv")

    df_all.to_csv(out_all, index=False)
    df_sum.to_csv(out_sum, index=False)
    df_ad.to_csv(out_ad, index=False)

    print("\nSaved:")
    print(" -", out_all)
    print(" -", out_sum)
    print(" -", out_ad)

    print("\nAblation summary (mean±sd):")
    print(df_sum)

    # Paired tests: baseline vs pseudo_w1 (overall + inAD)
    print("\n=== Paired fold statistics: baseline vs pseudo_w1 ===")
    for metric in ["mcc", "bal_acc", "roc_auc", "mcc_inAD", "bal_acc_inAD"]:
        st = paired_stats(df_all, model_a="baseline", model_b="pseudo_w1", metric=metric)
        print(
            f"{metric}: Δmean={st['delta_mean']:+.4f} "
            f"(95% CI {st['delta_ci_low']:+.4f} to {st['delta_ci_high']:+.4f}) | "
            f"t p={st['t_p']:.4g} | Wilcoxon p={st['w_p']:.4g} | dz={st['dz']:.3f}"
        )

    # AD coverage report
    cov = (df_all.groupby("model")["test_in_ad_frac"]
           .agg(["mean","std","min","max"])
           .reset_index())
    print("\nAD coverage (test_in_ad_frac):")
    print(cov)

    return df_all, df_sum, df_ad

# Notebook run:
df_all, df_sum, df_ad = main()


[UMAP] FIGDIR = C:\Users\Besitzer\Desktop\M3_databases\figures\UMAP_plots
[UMAP] FIGDIR exists: True
[UMAP] write test OK: C:\Users\Besitzer\Desktop\M3_databases\figures\UMAP_plots\_write_test.txt
Loaded: (2268, 17)
Label counts:
 consensus_label
active_single      1502
inactive_single     463
active              286
inactive             17
Name: count, dtype: int64


c:\Users\Besitzer\anaconda3\envs\openms_env\lib\site-packages\umap\umap_.py:1887: UserWarning: gradient function is not yet implemented for jaccard distance metric; inverse_transform will be unavailable
  warn(
c:\Users\Besitzer\anaconda3\envs\openms_env\lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


[UMAP] wrote: C:\Users\Besitzer\Desktop\M3_databases\figures\UMAP_plots\UMAP_global_activity.png | exists: True | bytes: 318308
[UMAP] wrote: C:\Users\Besitzer\Desktop\M3_databases\figures\UMAP_plots\UMAP_global_activity.pdf | exists: True | bytes: 48548
[UMAP] global done
[UMAP] fold plot done

=== A) Baseline (ChEMBL only) ===
[baseline Fold 1] running...
[baseline Fold 1] ROC=0.997 PR=0.999 MCC=0.883 BalAcc=0.925 | inAD=0.97
[baseline Fold 2] running...


c:\Users\Besitzer\anaconda3\envs\openms_env\lib\site-packages\sklearn\metrics\_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


[baseline Fold 2] ROC=0.986 PR=0.997 MCC=0.845 BalAcc=0.923 | inAD=0.96
[baseline Fold 3] running...
[baseline Fold 3] ROC=0.996 PR=0.999 MCC=0.875 BalAcc=0.924 | inAD=0.98
[baseline Fold 4] running...
[baseline Fold 4] ROC=0.981 PR=0.997 MCC=0.766 BalAcc=0.826 | inAD=0.95
[baseline Fold 5] running...
[baseline Fold 5] ROC=0.982 PR=0.996 MCC=0.821 BalAcc=0.925 | inAD=0.93
[baseline Fold 6] running...
[baseline Fold 6] ROC=0.988 PR=0.994 MCC=0.872 BalAcc=0.929 | inAD=0.90
[baseline Fold 7] running...
[baseline Fold 7] ROC=0.983 PR=0.990 MCC=0.851 BalAcc=0.917 | inAD=0.89
[baseline Fold 8] running...
[baseline Fold 8] ROC=0.986 PR=0.997 MCC=0.798 BalAcc=0.931 | inAD=0.96
[baseline Fold 9] running...
[baseline Fold 9] ROC=0.954 PR=0.992 MCC=0.626 BalAcc=0.859 | inAD=0.96
[baseline Fold 10] running...
[baseline Fold 10] ROC=0.972 PR=0.987 MCC=0.705 BalAcc=0.804 | inAD=0.89

=== B) Pseudo-neg augmentation (weight=1.0) ===
[pseudo_w1 Fold 1] train pos=1604 neg=437 need pseudo_neg=1167


...

[pseudo_w03 Fold 8] ROC=0.987 PR=0.998 MCC=0.798 BalAcc=0.931 | inAD=0.96
[pseudo_w03 Fold 9] train pos=1597 neg=445 need pseudo_neg=1152


[17:36:22] Can't kekulize mol.  Unkekulized atoms: 1 2 3 4 5 6 10 11 15 16 17 19 20 21
[17:36:22] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19


[pseudo_w03 Fold 9] ROC=0.953 PR=0.991 MCC=0.626 BalAcc=0.859 | inAD=0.96
[pseudo_w03 Fold 10] train pos=1628 neg=414 need pseudo_neg=1214


[17:39:02] Can't kekulize mol.  Unkekulized atoms: 1 2 3 4 5 6 10 11 15 16 17 19 20 21
[17:39:02] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19


[pseudo_w03 Fold 10] ROC=0.971 PR=0.986 MCC=0.739 BalAcc=0.827 | inAD=0.89

Saved:
 - C:\Users\Besitzer\Desktop\M3_databases\pseudo_neg_controls_scaffoldcv\scaffoldcv_ablation_perfold_with_AD.csv
 - C:\Users\Besitzer\Desktop\M3_databases\pseudo_neg_controls_scaffoldcv\scaffoldcv_ablation_summary_with_AD.csv
 - C:\Users\Besitzer\Desktop\M3_databases\pseudo_neg_controls_scaffoldcv\scaffoldcv_AD_diagnostics.csv

Ablation summary (mean±sd):
        model   eps  butina_cutoff_dist  pseudo_weight  ad_threshold  \
0    baseline   NaN                 NaN            NaN           0.4   
1  pseudo_w03  0.01                0.65            0.3           0.4   
2   pseudo_w1  0.01                0.65            1.0           0.4   

   pseudo_ad_threshold  roc_auc_mean  roc_auc_std  pr_auc_mean  pr_auc_std  \
0                  NaN      0.982474     0.012485     0.994847    0.004174   
1                  0.4      0.982437     0.012674     0.994826    0.004238   
2                  0.4      0.982423

In [ ]:
# ================================================================
# M3 scaffold-CV + pseudo-negative augmentation + Applicability Domain
# + Publication-grade UMAP plots (global + fold train vs test)
#
#   - ChEMBL train: consensus_label -> y in {0,1}
#   - Morgan FP (RDKit MorganGenerator): radius=2, nBits=2048
#   - GroupKFold by Murcko scaffold
#   - Pseudo-negatives streamed from COCONUT-A (scaffold-hash split)
#   - Butina clustering for diversity
#   - AD: max Tanimoto similarity to training fold
#   - Report: in-AD vs out-of-AD performance + coverage
#   - Statistics: paired tests + Cohen's dz + bootstrap CI
#   - UMAP: global + fold-specific train vs test separation plots
#
# Outputs (CV):
#   - scaffoldcv_ablation_perfold_with_AD.csv
#   - scaffoldcv_ablation_summary_with_AD.csv
#   - scaffoldcv_AD_diagnostics.csv
#
# Outputs (UMAP):
#   - UMAP_global_activity.png/.pdf
#   - UMAP_foldXX_train_vs_test.png/.pdf
# ================================================================

import os
import math
import hashlib
import numpy as np
import pandas as pd

from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import rdFingerprintGenerator
from rdkit.ML.Cluster import Butina

from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    balanced_accuracy_score,
)

from scipy.stats import ttest_rel, wilcoxon

# --- UMAP + plotting (install if needed: conda-forge umap-learn, or pip umap-learn) ---
import matplotlib.pyplot as plt
import umap

RDLogger.DisableLog("rdApp.warning")

# -------------------------
# PATHS (adjust if needed)
# -------------------------
BASE = r"C:\Users\Besitzer\Desktop\M3_databases"
TRAIN_CSV = os.path.join(BASE, "ChEMBL_M3_consensus_labels_more_negatives_with_meta.csv")
COCO_CSV  = os.path.join(BASE, "coconut_screen_out", "coconut_screen_ranked.csv")

OUTDIR_CV    = os.path.join(BASE, "pseudo_neg_controls_scaffoldcv")
OUTDIR_UMAP  = os.path.join(BASE, "figures", "UMAP_plots")

os.makedirs(OUTDIR_CV, exist_ok=True)
os.makedirs(OUTDIR_UMAP, exist_ok=True)

# -------------------------
# GLOBAL SETTINGS
# -------------------------
N_SPLITS = 10
RANDOM_STATE = 0

AD_THRESHOLD = 0.40
PSEUDO_AD_THRESHOLD = 0.40  # set None to disable pseudo-AD filter

# Pseudo-neg selection
EPS = 0.01
BUTINA_CUTOFF_DIST = 0.65
MAX_COCO_CANDIDATES = 20000
CHUNKSIZE = 20000

# Fingerprints
MORGAN_RADIUS = 2
MORGAN_NBITS  = 2048
_morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=MORGAN_RADIUS, fpSize=MORGAN_NBITS)

# -------------------------
# QUICK RUN SWITCHES
# -------------------------
RUN_CV_PIPELINE = True   # set False to skip CV completely
RUN_UMAP_PLOTS  = True   # set True to generate UMAP figures
UMAP_FOLD_ID    = 1      # fold for fold-specific plot

# UMAP params (good defaults for binary Morgan)
UMAP_N_NEIGHBORS = 15
UMAP_MIN_DIST    = 0.10
UMAP_METRIC      = "jaccard"   # recommended for 0/1 fingerprints
UMAP_POINT_SIZE  = 18


# ================================================================
# (1) Helpers: labels, mol parsing, scaffolds, fingerprints
# ================================================================

def label_to_binary(consensus_label: str):
    if consensus_label in ("active", "active_single"):
        return 1
    if consensus_label in ("inactive", "inactive_single"):
        return 0
    return None

def safe_mol_from_smiles(smiles: str):
    if not isinstance(smiles, str) or not smiles.strip():
        return None
    try:
        return Chem.MolFromSmiles(smiles)
    except Exception:
        return None

def murcko_scaffold_smiles(mol):
    try:
        scaf = MurckoScaffold.GetScaffoldForMol(mol)
        if scaf is None:
            return None
        return Chem.MolToSmiles(scaf, isomericSmiles=False)
    except Exception:
        return None

def mol_to_fp(mol):
    try:
        return _morgan_gen.GetFingerprint(mol)
    except Exception:
        return None

def fps_to_numpy(fps):
    X = np.zeros((len(fps), MORGAN_NBITS), dtype=np.uint8)
    for i, fp in enumerate(fps):
        arr = np.zeros((MORGAN_NBITS,), dtype=np.int8)
        DataStructs.ConvertToNumpyArray(fp, arr)
        X[i, :] = arr
    return X

def safe_binary_metrics(y_true, p_pred, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    p_pred = np.asarray(p_pred).astype(float)
    y_hat = (p_pred >= threshold).astype(int)

    out = {}
    if len(y_true) == 0:
        return {"roc_auc": np.nan, "pr_auc": np.nan, "mcc": np.nan, "bal_acc": np.nan}

    if np.unique(y_true).size < 2:
        out["roc_auc"] = np.nan
        out["pr_auc"]  = np.nan
    else:
        out["roc_auc"] = float(roc_auc_score(y_true, p_pred))
        out["pr_auc"]  = float(average_precision_score(y_true, p_pred))

    out["mcc"]     = float(matthews_corrcoef(y_true, y_hat))
    out["bal_acc"] = float(balanced_accuracy_score(y_true, y_hat))
    return out

def train_lr():
    return LogisticRegression(
        max_iter=4000,
        solver="lbfgs",
        n_jobs=1
    )

# ================================================================
# (2) Butina clustering (diversity control)
# ================================================================

def butina_cluster_fps(fps, cutoff_dist=0.65):
    if len(fps) == 0:
        return []
    if len(fps) == 1:
        return [[0]]

    dists = []
    for i in range(1, len(fps)):
        sims = DataStructs.BulkTanimotoSimilarity(fps[i], fps[:i])
        dists.extend([1.0 - s for s in sims])

    clusters = Butina.ClusterData(dists, len(fps), cutoff_dist, isDistData=True)
    return [list(c) for c in clusters]

# ================================================================
# (3) Load ChEMBL train, compute scaffolds + fingerprints
# ================================================================

def load_chembl_train(train_csv):
    df = pd.read_csv(train_csv)

    if "consensus_label" not in df.columns:
        raise ValueError("ChEMBL CSV must contain 'consensus_label' column.")

    df["y"] = df["consensus_label"].map(label_to_binary)
    df = df[df["y"].isin([0, 1])].copy()

    smiles_col = None
    for c in ["canonical_smiles", "smiles", "Smiles", "SMILES"]:
        if c in df.columns:
            smiles_col = c
            break
    if smiles_col is None:
        raise ValueError("Could not find a SMILES column in ChEMBL CSV.")

    df["mol"] = df[smiles_col].apply(safe_mol_from_smiles)
    df = df[df["mol"].notnull()].copy()

    df["scaffold"] = df["mol"].apply(murcko_scaffold_smiles)
    df = df[df["scaffold"].notnull()].copy()

    df["fp"] = df["mol"].apply(mol_to_fp)
    df = df[df["fp"].notnull()].copy()

    return df, smiles_col

# ================================================================
# (4) Deterministic scaffold split of COCONUT into A/B by hash
# ================================================================

def scaffold_to_half(scaffold: str, split_ratio=0.5):
    h = hashlib.md5(scaffold.encode("utf-8")).hexdigest()
    x = int(h[:8], 16) / float(16**8)
    return "A" if x < split_ratio else "B"

# ================================================================
# (5) Applicability Domain (similarity-based)
# ================================================================

def max_tanimoto_to_train(fp, train_fps):
    sims = DataStructs.BulkTanimotoSimilarity(fp, train_fps)
    return float(max(sims)) if sims else float("nan")

def compute_ad_for_fps(query_fps, train_fps):
    return np.array([max_tanimoto_to_train(fp, train_fps) for fp in query_fps], dtype=float)

def eval_metrics(y_true, p_score, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    p_score = np.asarray(p_score).astype(float)

    if len(y_true) == 0:
        return {"roc_auc": np.nan, "pr_auc": np.nan, "mcc": np.nan, "bal_acc": np.nan,
                "n": 0, "pos": 0, "neg": 0}

    y_hat = (p_score >= threshold).astype(int)
    out = {}

    if np.unique(y_true).size < 2:
        out["roc_auc"] = np.nan
        out["pr_auc"]  = np.nan
    else:
        out["roc_auc"] = float(roc_auc_score(y_true, p_score))
        out["pr_auc"]  = float(average_precision_score(y_true, p_score))

    out["mcc"]     = float(matthews_corrcoef(y_true, y_hat))
    out["bal_acc"] = float(balanced_accuracy_score(y_true, y_hat))

    out["n"]   = int(len(y_true))
    out["pos"] = int((y_true == 1).sum())
    out["neg"] = int((y_true == 0).sum())
    return out

# ================================================================
# (6) Stream COCONUT-A, score, and collect pseudo-negative candidates
# ================================================================

def collect_pseudo_neg_candidates(
    coco_csv,
    model,
    eps=0.01,
    max_candidates=20000,
    chunksize=20000,
    smiles_col_guess=("canonical_smiles", "smiles", "SMILES", "Smiles"),
):
    head = pd.read_csv(coco_csv, nrows=5)
    smi_col = None
    for c in smiles_col_guess:
        if c in head.columns:
            smi_col = c
            break
    if smi_col is None:
        raise ValueError("Could not find a SMILES column in COCONUT CSV.")

    kept = []

    for chunk in pd.read_csv(coco_csv, chunksize=chunksize):
        if smi_col not in chunk.columns:
            continue

        smiles_list = chunk[smi_col].astype(str).tolist()

        mols = [safe_mol_from_smiles(s) for s in smiles_list]
        scaffolds = [murcko_scaffold_smiles(m) if m is not None else None for m in mols]

        idx_ok = [i for i, sc in enumerate(scaffolds) if sc is not None]
        if not idx_ok:
            continue

        idx_A = [i for i in idx_ok if scaffold_to_half(scaffolds[i]) == "A"]
        if not idx_A:
            continue

        fps = []
        meta = []
        for i in idx_A:
            fp = mol_to_fp(mols[i])
            if fp is None:
                continue
            fps.append(fp)
            meta.append((smiles_list[i], scaffolds[i]))

        if not fps:
            continue

        X = fps_to_numpy(fps)
        p = model.predict_proba(X)[:, 1]

        for (smi, scaf), pi, fp in zip(meta, p, fps):
            if float(pi) <= eps:
                kept.append({"smiles": smi, "scaffold": scaf, "p": float(pi), "fp": fp})

        if len(kept) > max_candidates:
            kept.sort(key=lambda d: d["p"])
            kept = kept[:max_candidates]

    kept.sort(key=lambda d: d["p"])
    return kept

def select_diverse_pseudo_negs(candidates, n_needed, cutoff_dist=0.65):
    if n_needed <= 0 or len(candidates) == 0:
        return []

    fps = [d["fp"] for d in candidates]
    clusters = butina_cluster_fps(fps, cutoff_dist=cutoff_dist)

    selected = []
    for cl in clusters:
        best_idx = min(cl, key=lambda idx: candidates[idx]["p"])
        selected.append(candidates[best_idx])

    selected.sort(key=lambda d: d["p"])
    return selected[:n_needed]

# ================================================================
# (7) One fold runner: baseline or pseudo-neg + AD
# ================================================================

def run_fold_with_ad(
    X_tr, y_tr, fp_tr,
    X_te, y_te, fp_te,
    pseudo_weight=None,
    eps=0.01,
    butina_cutoff_dist=0.65,
    max_coco_candidates=20000,
    ad_threshold=0.40,
    pseudo_ad_threshold=0.40,
):
    # 1) Train baseline on training fold
    base_model = train_lr()
    base_model.fit(X_tr, y_tr)

    # 2) Optional pseudo-neg augmentation
    pseudo = []
    pseudo_sim = None

    if pseudo_weight is not None:
        n_pos = int((y_tr == 1).sum())
        n_neg = int((y_tr == 0).sum())
        n_needed = max(0, n_pos - n_neg)

        candidates = collect_pseudo_neg_candidates(
            COCO_CSV,
            model=base_model,
            eps=eps,
            max_candidates=max_coco_candidates,
            chunksize=CHUNKSIZE,
        )

        pseudo = select_diverse_pseudo_negs(
            candidates,
            n_needed=n_needed,
            cutoff_dist=butina_cutoff_dist
        )

        # AD filter for pseudo-negs
        if pseudo_ad_threshold is not None and len(pseudo) > 0:
            pseudo_fps = [d["fp"] for d in pseudo]
            pseudo_sim = compute_ad_for_fps(pseudo_fps, fp_tr)
            keep_mask = pseudo_sim >= float(pseudo_ad_threshold)
            pseudo = [d for d, keep in zip(pseudo, keep_mask) if keep]
            pseudo_sim = pseudo_sim[keep_mask] if pseudo_sim is not None else None

        if len(pseudo) > 0:
            X_pseudo = fps_to_numpy([d["fp"] for d in pseudo])
            y_pseudo = np.zeros((len(pseudo),), dtype=int)

            X_aug = np.vstack([X_tr, X_pseudo])
            y_aug = np.concatenate([y_tr, y_pseudo])

            w = np.ones((len(y_aug),), dtype=float)
            w[len(y_tr):] = float(pseudo_weight)
        else:
            X_aug, y_aug, w = X_tr, y_tr, None

        model = train_lr()
        if w is None:
            model.fit(X_aug, y_aug)
        else:
            model.fit(X_aug, y_aug, sample_weight=w)
    else:
        model = base_model

    # 3) Predict on test
    p_te = model.predict_proba(X_te)[:, 1]

    # 4) AD computation on test fold
    te_max_sim = compute_ad_for_fps(fp_te, fp_tr)
    in_ad_mask = te_max_sim >= float(ad_threshold)
    out_ad_mask = ~in_ad_mask

    # 5) Metrics overall + stratified
    overall = eval_metrics(y_te, p_te)
    in_ad   = eval_metrics(y_te[in_ad_mask],  p_te[in_ad_mask])  if in_ad_mask.any()  else eval_metrics([], [])
    out_ad  = eval_metrics(y_te[out_ad_mask], p_te[out_ad_mask]) if out_ad_mask.any() else eval_metrics([], [])

    # 6) Coverage + similarity summary
    ad_stats = {
        "ad_threshold": float(ad_threshold),
        "test_in_ad_frac": float(in_ad_mask.mean()),
        "test_in_ad_n": int(in_ad_mask.sum()),
        "test_out_ad_n": int(out_ad_mask.sum()),
        "test_maxsim_mean": float(np.nanmean(te_max_sim)),
        "test_maxsim_median": float(np.nanmedian(te_max_sim)),
        "test_maxsim_p10": float(np.nanpercentile(te_max_sim, 10)),
        "test_maxsim_p90": float(np.nanpercentile(te_max_sim, 90)),
    }

    # 7) Pseudo-neg diagnostics
    pseudo_stats = {
        "n_pseudo": int(len(pseudo)),
        "pseudo_weight": float(pseudo_weight) if pseudo_weight is not None else np.nan,
        "eps": float(eps) if pseudo_weight is not None else np.nan,
        "butina_cutoff_dist": float(butina_cutoff_dist) if pseudo_weight is not None else np.nan,
        "pseudo_ad_threshold": float(pseudo_ad_threshold) if (pseudo_weight is not None and pseudo_ad_threshold is not None) else np.nan,
        "pseudo_maxsim_mean": float(np.nanmean(pseudo_sim)) if (pseudo_sim is not None and len(pseudo_sim) > 0) else np.nan,
        "pseudo_maxsim_median": float(np.nanmedian(pseudo_sim)) if (pseudo_sim is not None and len(pseudo_sim) > 0) else np.nan,
        "pseudo_maxsim_p10": float(np.nanpercentile(pseudo_sim, 10)) if (pseudo_sim is not None and len(pseudo_sim) > 0) else np.nan,
        "pseudo_maxsim_p90": float(np.nanpercentile(pseudo_sim, 90)) if (pseudo_sim is not None and len(pseudo_sim) > 0) else np.nan,
    }

    return overall, in_ad, out_ad, ad_stats, pseudo_stats

# ================================================================
# (8) CV runners
# ================================================================

def run_scaffoldcv(chembl_df, model_name, pseudo_weight=None, eps=0.01, butina_cutoff_dist=0.65,
                   max_coco_candidates=20000, n_splits=10, ad_threshold=0.40, pseudo_ad_threshold=0.40):

    y = chembl_df["y"].values.astype(int)
    scaff = chembl_df["scaffold"].values
    fp_all = chembl_df["fp"].tolist()
    X = fps_to_numpy(fp_all)

    gkf = GroupKFold(n_splits=n_splits)

    rows = []
    ad_rows = []

    for fold, (tr, te) in enumerate(gkf.split(X, y, groups=scaff), start=1):
        X_tr, y_tr = X[tr], y[tr]
        X_te, y_te = X[te], y[te]
        fp_tr = [fp_all[i] for i in tr]
        fp_te = [fp_all[i] for i in te]

        if pseudo_weight is None:
            print(f"[{model_name} Fold {fold}] running...")
        else:
            n_pos = int((y_tr == 1).sum())
            n_neg = int((y_tr == 0).sum())
            n_need = max(0, n_pos - n_neg)
            print(f"[{model_name} Fold {fold}] train pos={n_pos} neg={n_neg} need pseudo_neg={n_need}")

        overall, in_ad, out_ad, ad_stats, pseudo_stats = run_fold_with_ad(
            X_tr=X_tr, y_tr=y_tr, fp_tr=fp_tr,
            X_te=X_te, y_te=y_te, fp_te=fp_te,
            pseudo_weight=pseudo_weight,
            eps=eps,
            butina_cutoff_dist=butina_cutoff_dist,
            max_coco_candidates=max_coco_candidates,
            ad_threshold=ad_threshold,
            pseudo_ad_threshold=pseudo_ad_threshold,
        )

        row = {
            "model": model_name,
            "fold": fold,
            "n_train_pos": int((y_tr == 1).sum()),
            "n_train_neg": int((y_tr == 0).sum()),
            **pseudo_stats,

            "roc_auc": overall["roc_auc"],
            "pr_auc": overall["pr_auc"],
            "mcc": overall["mcc"],
            "bal_acc": overall["bal_acc"],

            "roc_auc_inAD": in_ad["roc_auc"],
            "pr_auc_inAD": in_ad["pr_auc"],
            "mcc_inAD": in_ad["mcc"],
            "bal_acc_inAD": in_ad["bal_acc"],
            "n_inAD": in_ad["n"],

            "roc_auc_outAD": out_ad["roc_auc"],
            "pr_auc_outAD": out_ad["pr_auc"],
            "mcc_outAD": out_ad["mcc"],
            "bal_acc_outAD": out_ad["bal_acc"],
            "n_outAD": out_ad["n"],

            **ad_stats,
        }
        rows.append(row)

        print(f"[{model_name} Fold {fold}] ROC={overall['roc_auc']:.3f} PR={overall['pr_auc']:.3f} "
              f"MCC={overall['mcc']:.3f} BalAcc={overall['bal_acc']:.3f} | inAD={ad_stats['test_in_ad_frac']:.2f}")

        # AD diagnostics table (per fold) — FIXED (no undefined m_all/m_in/m_out)
        ad_rows.append({
            "model": model_name,
            "fold": fold,
            **ad_stats,
            **{k: pseudo_stats[k] for k in [
                "n_pseudo", "pseudo_weight", "eps", "butina_cutoff_dist", "pseudo_ad_threshold",
                "pseudo_maxsim_mean", "pseudo_maxsim_median", "pseudo_maxsim_p10", "pseudo_maxsim_p90"
            ]},

            "roc_auc": overall["roc_auc"],
            "pr_auc":  overall["pr_auc"],
            "mcc":     overall["mcc"],
            "bal_acc": overall["bal_acc"],

            "roc_auc_inAD": in_ad["roc_auc"],
            "pr_auc_inAD":  in_ad["pr_auc"],
            "mcc_inAD":     in_ad["mcc"],
            "bal_acc_inAD": in_ad["bal_acc"],
            "n_inAD":       in_ad["n"],

            "roc_auc_outAD": out_ad["roc_auc"],
            "pr_auc_outAD":  out_ad["pr_auc"],
            "mcc_outAD":     out_ad["mcc"],
            "bal_acc_outAD": out_ad["bal_acc"],
            "n_outAD":       out_ad["n"],
        })

    df = pd.DataFrame(rows)
    df_ad = pd.DataFrame(ad_rows)
    return df, df_ad

def summarize_runs(df_all, metrics):
    group_cols = ["model", "eps", "butina_cutoff_dist", "pseudo_weight", "ad_threshold", "pseudo_ad_threshold"]
    out = (df_all
           .groupby(group_cols, dropna=False)[metrics]
           .agg(["mean", "std"])
           .reset_index())
    out.columns = ["_".join([c for c in col if c]) if isinstance(col, tuple) else col for col in out.columns]
    return out

# ================================================================
# (9) Statistics helpers (paired folds)
# ================================================================

def paired_stats(df_all, model_a, model_b, metric="mcc"):
    A = df_all[df_all["model"] == model_a][["fold", metric]].rename(columns={metric: f"{metric}_A"})
    B = df_all[df_all["model"] == model_b][["fold", metric]].rename(columns={metric: f"{metric}_B"})
    paired = A.merge(B, on="fold", how="inner").sort_values("fold")

    x = paired[f"{metric}_A"].astype(float).values
    y = paired[f"{metric}_B"].astype(float).values

    d = (y - x)
    d = d[np.isfinite(d)]
    n = len(d)

    out = {"metric": metric, "n": int(n)}

    if n < 2:
        out.update({
            "t_p": np.nan, "t_stat": np.nan, "w_p": np.nan, "w_stat": np.nan, "dz": np.nan,
            "delta_mean": float(np.nanmean(d)) if n == 1 else np.nan,
            "delta_ci_low": np.nan, "delta_ci_high": np.nan
        })
        return out

    t = ttest_rel(x, y, nan_policy="omit")
    out["t_stat"] = float(t.statistic)
    out["t_p"] = float(t.pvalue)

    try:
        w = wilcoxon(x, y)
        out["w_stat"] = float(w.statistic)
        out["w_p"] = float(w.pvalue)
    except Exception:
        out["w_stat"] = np.nan
        out["w_p"] = np.nan

    sd = float(np.std(d, ddof=1))
    out["delta_mean"] = float(np.mean(d))
    out["delta_sd"] = sd
    out["dz"] = float(out["delta_mean"] / sd) if sd > 0 else np.nan

    rng = np.random.default_rng(RANDOM_STATE)
    Bn = 5000
    boots = np.array([np.mean(rng.choice(d, size=n, replace=True)) for _ in range(Bn)], dtype=float)
    out["delta_ci_low"] = float(np.percentile(boots, 2.5))
    out["delta_ci_high"] = float(np.percentile(boots, 97.5))

    return out

# ================================================================
# (10) Ablation suite + AD
# ================================================================

def run_ablation_suite_with_ad(chembl_df):
    print("\n=== A) Baseline (ChEMBL only) ===")
    df_base, df_base_ad = run_scaffoldcv(
        chembl_df,
        model_name="baseline",
        pseudo_weight=None,
        n_splits=N_SPLITS,
        ad_threshold=AD_THRESHOLD,
        pseudo_ad_threshold=PSEUDO_AD_THRESHOLD,
    )

    print("\n=== B) Pseudo-neg augmentation (weight=1.0) ===")
    df_w1, df_w1_ad = run_scaffoldcv(
        chembl_df,
        model_name="pseudo_w1",
        pseudo_weight=1.0,
        eps=EPS,
        butina_cutoff_dist=BUTINA_CUTOFF_DIST,
        max_coco_candidates=MAX_COCO_CANDIDATES,
        n_splits=N_SPLITS,
        ad_threshold=AD_THRESHOLD,
        pseudo_ad_threshold=PSEUDO_AD_THRESHOLD,
    )

    print("\n=== C) Pseudo-neg augmentation (weight=0.3) ===")
    df_w03, df_w03_ad = run_scaffoldcv(
        chembl_df,
        model_name="pseudo_w03",
        pseudo_weight=0.3,
        eps=EPS,
        butina_cutoff_dist=BUTINA_CUTOFF_DIST,
        max_coco_candidates=MAX_COCO_CANDIDATES,
        n_splits=N_SPLITS,
        ad_threshold=AD_THRESHOLD,
        pseudo_ad_threshold=PSEUDO_AD_THRESHOLD,
    )

    df_all = pd.concat([df_base, df_w1, df_w03], ignore_index=True)
    df_ad  = pd.concat([df_base_ad, df_w1_ad, df_w03_ad], ignore_index=True)

    metrics = [
        "roc_auc","pr_auc","mcc","bal_acc",
        "roc_auc_inAD","pr_auc_inAD","mcc_inAD","bal_acc_inAD",
        "roc_auc_outAD","pr_auc_outAD","mcc_outAD","bal_acc_outAD",
        "test_in_ad_frac","test_maxsim_mean","test_maxsim_median",
        "n_pseudo","pseudo_maxsim_mean","pseudo_maxsim_median"
    ]
    df_sum = summarize_runs(df_all, metrics=metrics)

    return df_all, df_sum, df_ad

# ================================================================
# (11) UMAP plotting (publication-grade matplotlib)
# ================================================================

def _save_fig(fig, out_prefix):
    fig.savefig(out_prefix + ".png", dpi=300, bbox_inches="tight")
    fig.savefig(out_prefix + ".pdf", bbox_inches="tight")
    plt.close(fig)
    print("[SAVED]", out_prefix + ".png")
    print("[SAVED]", out_prefix + ".pdf")

def plot_umap_global(
    chembl_df,
    out_prefix,
    n_neighbors=15,
    min_dist=0.10,
    metric="jaccard",
    random_state=0,
    point_size=18,
):
    fps = chembl_df["fp"].tolist()
    X = fps_to_numpy(fps).astype(np.uint8)
    y = chembl_df["y"].values.astype(int)

    reducer = umap.UMAP(
        n_neighbors=int(n_neighbors),
        min_dist=float(min_dist),
        metric=str(metric),
        random_state=int(random_state),
        n_components=2,
    )

    emb = reducer.fit_transform(X)

    fig = plt.figure(figsize=(8.2, 6.6))
    ax = plt.gca()

    mask0 = (y == 0)
    mask1 = (y == 1)

    ax.scatter(emb[mask0, 0], emb[mask0, 1], s=point_size, alpha=0.85, label="Inactive (0)")
    ax.scatter(emb[mask1, 0], emb[mask1, 1], s=point_size, alpha=0.85, label="Active (1)")

    ax.set_title("UMAP of ChEMBL M3 (Morgan FP)", fontsize=18)
    ax.set_xlabel("UMAP-1", fontsize=14)
    ax.set_ylabel("UMAP-2", fontsize=14)
    ax.tick_params(labelsize=12)
    ax.legend(frameon=False, fontsize=12, loc="upper right")

    _save_fig(fig, out_prefix)

def plot_umap_fold_train_test(
    chembl_df,
    out_prefix,
    fold_id=1,
    n_splits=10,
    n_neighbors=15,
    min_dist=0.10,
    metric="jaccard",
    random_state=0,
    point_size=18,
):
    y = chembl_df["y"].values.astype(int)
    scaff = chembl_df["scaffold"].values
    fps = chembl_df["fp"].tolist()
    X = fps_to_numpy(fps).astype(np.uint8)

    gkf = GroupKFold(n_splits=int(n_splits))
    splits = list(gkf.split(X, y, groups=scaff))
    if fold_id < 1 or fold_id > len(splits):
        raise ValueError(f"fold_id must be in [1, {len(splits)}], got {fold_id}")

    tr, te = splits[fold_id - 1]

    X_tr, y_tr = X[tr], y[tr]
    X_te, y_te = X[te], y[te]

    reducer = umap.UMAP(
        n_neighbors=int(n_neighbors),
        min_dist=float(min_dist),
        metric=str(metric),
        random_state=int(random_state),
        n_components=2,
    )

    emb_tr = reducer.fit_transform(X_tr)
    emb_te = reducer.transform(X_te)

    fig = plt.figure(figsize=(9.2, 7.0))
    ax = plt.gca()

    # Train: circles, Test: triangles
    tr0 = (y_tr == 0)
    tr1 = (y_tr == 1)
    te0 = (y_te == 0)
    te1 = (y_te == 1)

    ax.scatter(emb_tr[tr0, 0], emb_tr[tr0, 1], s=point_size, alpha=0.60, marker="o", label="Train inactive")
    ax.scatter(emb_tr[tr1, 0], emb_tr[tr1, 1], s=point_size, alpha=0.60, marker="o", label="Train active")

    ax.scatter(emb_te[te0, 0], emb_te[te0, 1], s=point_size * 1.15, alpha=0.95, marker="^", label="Test inactive")
    ax.scatter(emb_te[te1, 0], emb_te[te1, 1], s=point_size * 1.15, alpha=0.95, marker="^", label="Test active")

    ax.set_title(f"UMAP fold {fold_id}: train vs test (fit on train)", fontsize=18)
    ax.set_xlabel("UMAP-1", fontsize=14)
    ax.set_ylabel("UMAP-2", fontsize=14)
    ax.tick_params(labelsize=12)
    ax.legend(frameon=False, fontsize=12, loc="lower left")

    _save_fig(fig, out_prefix)

# ================================================================
# (12) Main
# ================================================================

def main():
    chembl_df, smiles_col = load_chembl_train(TRAIN_CSV)

    print("Loaded:", chembl_df.shape)
    print("Label counts:\n", chembl_df["consensus_label"].value_counts())

    # --- UMAP plots (can be run standalone) ---
    if RUN_UMAP_PLOTS:
        plot_umap_global(
            chembl_df,
            out_prefix=os.path.join(OUTDIR_UMAP, "UMAP_global_activity"),
            n_neighbors=UMAP_N_NEIGHBORS,
            min_dist=UMAP_MIN_DIST,
            metric=UMAP_METRIC,
            random_state=RANDOM_STATE,
            point_size=UMAP_POINT_SIZE,
        )

        plot_umap_fold_train_test(
            chembl_df,
            out_prefix=os.path.join(OUTDIR_UMAP, f"UMAP_fold{UMAP_FOLD_ID:02d}_train_vs_test"),
            fold_id=UMAP_FOLD_ID,
            n_splits=N_SPLITS,
            n_neighbors=UMAP_N_NEIGHBORS,
            min_dist=UMAP_MIN_DIST,
            metric=UMAP_METRIC,
            random_state=RANDOM_STATE,
            point_size=UMAP_POINT_SIZE,
        )

    # --- CV pipeline (optional) ---
    if RUN_CV_PIPELINE:
        df_all, df_sum, df_ad = run_ablation_suite_with_ad(chembl_df)

        out_all = os.path.join(OUTDIR_CV, "scaffoldcv_ablation_perfold_with_AD.csv")
        out_sum = os.path.join(OUTDIR_CV, "scaffoldcv_ablation_summary_with_AD.csv")
        out_ad  = os.path.join(OUTDIR_CV, "scaffoldcv_AD_diagnostics.csv")

        df_all.to_csv(out_all, index=False)
        df_sum.to_csv(out_sum, index=False)
        df_ad.to_csv(out_ad, index=False)

        print("\nSaved:")
        print(" -", out_all)
        print(" -", out_sum)
        print(" -", out_ad)

        print("\nAblation summary (mean±sd):")
        print(df_sum)

        print("\n=== Paired fold statistics: baseline vs pseudo_w1 ===")
        for metric in ["mcc", "bal_acc", "roc_auc", "mcc_inAD", "bal_acc_inAD"]:
            st = paired_stats(df_all, model_a="baseline", model_b="pseudo_w1", metric=metric)
            print(f"{metric}: Δmean={st['delta_mean']:+.4f} "
                  f"(95% CI {st['delta_ci_low']:+.4f} to {st['delta_ci_high']:+.4f}) | "
                  f"t p={st['t_p']:.4g} | Wilcoxon p={st['w_p']:.4g} | dz={st['dz']:.3f}")

        cov = (df_all.groupby("model")["test_in_ad_frac"]
               .agg(["mean","std","min","max"])
               .reset_index())
        print("\nAD coverage (test_in_ad_frac):")
        print(cov)

        return chembl_df, df_all, df_sum, df_ad

    return chembl_df, None, None, None


# Notebook run:
chembl_df, df_all, df_sum, df_ad = main()


In [3]:
# ================================================================
# M3 scaffold-CV + pseudo-negative augmentation + Applicability Domain
#   + UMAP plotting (publication-ready matplotlib)
#
#   - ChEMBL train: consensus_label -> y in {0,1}
#   - Morgan FP (RDKit MorganGenerator): radius=2, nBits=2048
#   - GroupKFold by Murcko scaffold
#   - Pseudo-negatives streamed from COCONUT-A (scaffold-hash split)
#   - Butina clustering for diversity
#   - AD: max Tanimoto similarity to training fold
#   - Report: in-AD vs out-of-AD performance + coverage
#   - Statistics: paired tests + Cohen's dz + bootstrap CI
#   - UMAP: global activity + fold train/test (fit on train, transform test)
#
# Outputs:
#   - scaffoldcv_ablation_perfold_with_AD.csv
#   - scaffoldcv_ablation_summary_with_AD.csv
#   - scaffoldcv_AD_diagnostics.csv
#   - figures/UMAP_plots/*.png + *.pdf
# ================================================================

import os
import math
import hashlib
import numpy as np
import pandas as pd

from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import rdFingerprintGenerator
from rdkit.ML.Cluster import Butina

from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    balanced_accuracy_score,
)

from scipy.stats import ttest_rel, wilcoxon

import matplotlib.pyplot as plt

RDLogger.DisableLog("rdApp.warning")


# -------------------------
# PATHS (adjust if needed)
# -------------------------
BASE = r"C:\Users\Besitzer\Desktop\M3_databases"
TRAIN_CSV = os.path.join(BASE, "ChEMBL_M3_consensus_labels_more_negatives_with_meta.csv")
COCO_CSV  = os.path.join(BASE, "coconut_screen_out", "coconut_screen_ranked.csv")
OUTDIR    = os.path.join(BASE, "pseudo_neg_controls_scaffoldcv")
os.makedirs(OUTDIR, exist_ok=True)


# -------------------------
# GLOBAL SETTINGS (publication knobs)
# -------------------------
N_SPLITS = 10
RANDOM_STATE = 0              # used for bootstrap only (GroupKFold is deterministic)
AD_THRESHOLD = 0.40           # similarity threshold for "in domain" on test fold
PSEUDO_AD_THRESHOLD = 0.40    # require pseudo-negs to be within AD of training fold (set None to disable)

# Pseudo-neg selection
EPS = 0.01
BUTINA_CUTOFF_DIST = 0.65
MAX_COCO_CANDIDATES = 20000
CHUNKSIZE = 20000

# Fingerprints
MORGAN_RADIUS = 2
MORGAN_NBITS  = 2048
_morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=MORGAN_RADIUS, fpSize=MORGAN_NBITS)


# ================================================================
# (1) Helpers: labels, mol parsing, scaffolds, fingerprints
# ================================================================

def label_to_binary(consensus_label: str):
    if consensus_label in ("active", "active_single"):
        return 1
    if consensus_label in ("inactive", "inactive_single"):
        return 0
    return None

def safe_mol_from_smiles(smiles: str):
    if not isinstance(smiles, str) or not smiles.strip():
        return None
    try:
        return Chem.MolFromSmiles(smiles)
    except Exception:
        return None

def murcko_scaffold_smiles(mol):
    try:
        scaf = MurckoScaffold.GetScaffoldForMol(mol)
        if scaf is None:
            return None
        return Chem.MolToSmiles(scaf, isomericSmiles=False)
    except Exception:
        return None

def mol_to_fp(mol):
    try:
        return _morgan_gen.GetFingerprint(mol)  # ExplicitBitVect-like
    except Exception:
        return None

def fps_to_numpy(fps):
    # scikit-learn expects numeric matrix; ConvertToNumpyArray gives 0/1 array
    X = np.zeros((len(fps), MORGAN_NBITS), dtype=np.uint8)
    for i, fp in enumerate(fps):
        arr = np.zeros((MORGAN_NBITS,), dtype=np.int8)
        DataStructs.ConvertToNumpyArray(fp, arr)
        X[i, :] = arr
    return X

def safe_binary_metrics(y_true, p_pred, threshold=0.5):
    """
    Compute ROC-AUC / PR-AUC / MCC / BalAcc safely.
    Returns dict with NaN for ROC/PR if y_true has only one class.
    """
    y_true = np.asarray(y_true).astype(int)
    p_pred = np.asarray(p_pred).astype(float)

    y_hat = (p_pred >= threshold).astype(int)

    out = {}
    if len(y_true) == 0:
        return {"roc_auc": np.nan, "pr_auc": np.nan, "mcc": np.nan, "bal_acc": np.nan}

    if np.unique(y_true).size < 2:
        out["roc_auc"] = np.nan
        out["pr_auc"]  = np.nan
    else:
        out["roc_auc"] = float(roc_auc_score(y_true, p_pred))
        out["pr_auc"]  = float(average_precision_score(y_true, p_pred))

    out["mcc"]     = float(matthews_corrcoef(y_true, y_hat))
    out["bal_acc"] = float(balanced_accuracy_score(y_true, y_hat))
    return out

def eval_metrics(y_true, p_score, threshold=0.5):
    """
    Same as safe_binary_metrics but also returns n/pos/neg for reporting.
    """
    y_true = np.asarray(y_true).astype(int)
    p_score = np.asarray(p_score).astype(float)

    if len(y_true) == 0:
        return {"roc_auc": np.nan, "pr_auc": np.nan, "mcc": np.nan, "bal_acc": np.nan,
                "n": 0, "pos": 0, "neg": 0}

    m = safe_binary_metrics(y_true, p_score, threshold=threshold)
    m["n"] = int(len(y_true))
    m["pos"] = int((y_true == 1).sum())
    m["neg"] = int((y_true == 0).sum())
    return m

def train_lr():
    return LogisticRegression(
        max_iter=4000,
        solver="lbfgs",
        n_jobs=1
    )


# ================================================================
# (2) Butina clustering (diversity control)
# ================================================================

def butina_cluster_fps(fps, cutoff_dist=0.65):
    if len(fps) == 0:
        return []
    if len(fps) == 1:
        return [[0]]

    dists = []
    for i in range(1, len(fps)):
        sims = DataStructs.BulkTanimotoSimilarity(fps[i], fps[:i])
        dists.extend([1.0 - s for s in sims])

    clusters = Butina.ClusterData(dists, len(fps), cutoff_dist, isDistData=True)
    return [list(c) for c in clusters]


# ================================================================
# (3) Load ChEMBL train, compute scaffolds + fingerprints
# ================================================================

def load_chembl_train(train_csv):
    df = pd.read_csv(train_csv)

    if "consensus_label" not in df.columns:
        raise ValueError("ChEMBL CSV must contain 'consensus_label' column.")

    df["y"] = df["consensus_label"].map(label_to_binary)
    df = df[df["y"].isin([0, 1])].copy()

    smiles_col = None
    for c in ["canonical_smiles", "smiles", "Smiles", "SMILES"]:
        if c in df.columns:
            smiles_col = c
            break
    if smiles_col is None:
        raise ValueError("Could not find a SMILES column in ChEMBL CSV.")

    df["mol"] = df[smiles_col].apply(safe_mol_from_smiles)
    df = df[df["mol"].notnull()].copy()

    df["scaffold"] = df["mol"].apply(murcko_scaffold_smiles)
    df = df[df["scaffold"].notnull()].copy()

    df["fp"] = df["mol"].apply(mol_to_fp)
    df = df[df["fp"].notnull()].copy()

    return df, smiles_col


# ================================================================
# (4) Deterministic scaffold split of COCONUT into A/B by hash
# ================================================================

def scaffold_to_half(scaffold: str, split_ratio=0.5):
    h = hashlib.md5(scaffold.encode("utf-8")).hexdigest()
    x = int(h[:8], 16) / float(16**8)
    return "A" if x < split_ratio else "B"


# ================================================================
# (5) Applicability Domain functions (similarity-based)
# ================================================================

def max_tanimoto_to_train(fp, train_fps):
    sims = DataStructs.BulkTanimotoSimilarity(fp, train_fps)
    return float(max(sims)) if sims else float("nan")

def compute_ad_for_fps(query_fps, train_fps):
    return np.array([max_tanimoto_to_train(fp, train_fps) for fp in query_fps], dtype=float)


# ================================================================
# (6) Stream COCONUT-A, score, and collect pseudo-negative candidates
# ================================================================

def collect_pseudo_neg_candidates(
    coco_csv,
    model,
    eps=0.01,
    max_candidates=20000,
    chunksize=20000,
    smiles_col_guess=("canonical_smiles", "smiles", "SMILES", "Smiles"),
):
    head = pd.read_csv(coco_csv, nrows=5)
    smi_col = None
    for c in smiles_col_guess:
        if c in head.columns:
            smi_col = c
            break
    if smi_col is None:
        raise ValueError("Could not find a SMILES column in COCONUT CSV.")

    kept = []

    for chunk in pd.read_csv(coco_csv, chunksize=chunksize):
        if smi_col not in chunk.columns:
            continue

        smiles_list = chunk[smi_col].astype(str).tolist()
        mols = [safe_mol_from_smiles(s) for s in smiles_list]
        scaffolds = [murcko_scaffold_smiles(m) if m is not None else None for m in mols]

        idx_ok = [i for i, sc in enumerate(scaffolds) if sc is not None]
        if not idx_ok:
            continue

        idx_A = [i for i in idx_ok if scaffold_to_half(scaffolds[i]) == "A"]
        if not idx_A:
            continue

        fps = []
        meta = []
        for i in idx_A:
            fp = mol_to_fp(mols[i])
            if fp is None:
                continue
            fps.append(fp)
            meta.append((smiles_list[i], scaffolds[i]))

        if not fps:
            continue

        X = fps_to_numpy(fps)
        p = model.predict_proba(X)[:, 1]

        for (smi, scaf), pi, fp in zip(meta, p, fps):
            if float(pi) <= eps:
                kept.append({"smiles": smi, "scaffold": scaf, "p": float(pi), "fp": fp})

        if len(kept) > max_candidates:
            kept.sort(key=lambda d: d["p"])
            kept = kept[:max_candidates]

    kept.sort(key=lambda d: d["p"])
    return kept

def select_diverse_pseudo_negs(candidates, n_needed, cutoff_dist=0.65):
    if n_needed <= 0 or len(candidates) == 0:
        return []

    fps = [d["fp"] for d in candidates]
    clusters = butina_cluster_fps(fps, cutoff_dist=cutoff_dist)

    selected = []
    for cl in clusters:
        best_idx = min(cl, key=lambda idx: candidates[idx]["p"])
        selected.append(candidates[best_idx])

    selected.sort(key=lambda d: d["p"])
    return selected[:n_needed]


# ================================================================
# (7) One fold runner: baseline or pseudo-neg + AD
# ================================================================

def run_fold_with_ad(
    X_tr, y_tr, fp_tr,
    X_te, y_te, fp_te,
    pseudo_weight=None,
    eps=0.01,
    butina_cutoff_dist=0.65,
    max_coco_candidates=20000,
    ad_threshold=0.40,
    pseudo_ad_threshold=0.40,
):
    """
    Returns:
      overall_metrics, inAD_metrics, outAD_metrics, ad_stats, pseudo_stats
    """
    # 1) Train baseline model on training fold
    base_model = train_lr()
    base_model.fit(X_tr, y_tr)

    # 2) Optional pseudo-neg augmentation
    pseudo = []
    pseudo_sim = None

    if pseudo_weight is not None:
        n_pos = int((y_tr == 1).sum())
        n_neg = int((y_tr == 0).sum())
        n_needed = max(0, n_pos - n_neg)

        candidates = collect_pseudo_neg_candidates(
            COCO_CSV,
            model=base_model,
            eps=eps,
            max_candidates=max_coco_candidates,
            chunksize=CHUNKSIZE,
        )

        pseudo = select_diverse_pseudo_negs(
            candidates,
            n_needed=n_needed,
            cutoff_dist=butina_cutoff_dist
        )

        # AD filter for pseudo-negs
        if pseudo_ad_threshold is not None and len(pseudo) > 0:
            pseudo_fps = [d["fp"] for d in pseudo]
            pseudo_sim = compute_ad_for_fps(pseudo_fps, fp_tr)
            keep_mask = pseudo_sim >= float(pseudo_ad_threshold)
            pseudo = [d for d, keep in zip(pseudo, keep_mask) if keep]
            pseudo_sim = pseudo_sim[keep_mask] if pseudo_sim is not None else None

        # Build augmented training set
        if len(pseudo) > 0:
            X_pseudo = fps_to_numpy([d["fp"] for d in pseudo])
            y_pseudo = np.zeros((len(pseudo),), dtype=int)

            X_aug = np.vstack([X_tr, X_pseudo])
            y_aug = np.concatenate([y_tr, y_pseudo])

            w = np.ones((len(y_aug),), dtype=float)
            w[len(y_tr):] = float(pseudo_weight)
        else:
            X_aug, y_aug, w = X_tr, y_tr, None

        # Retrain final model
        model = train_lr()
        if w is None:
            model.fit(X_aug, y_aug)
        else:
            model.fit(X_aug, y_aug, sample_weight=w)
    else:
        model = base_model

    # 3) Predict on test fold
    p_te = model.predict_proba(X_te)[:, 1]

    # 4) AD computation on test fold
    te_max_sim = compute_ad_for_fps(fp_te, fp_tr)
    in_ad_mask = te_max_sim >= float(ad_threshold)
    out_ad_mask = ~in_ad_mask

    # 5) Metrics overall + stratified
    overall = eval_metrics(y_te, p_te)
    in_ad   = eval_metrics(y_te[in_ad_mask],  p_te[in_ad_mask])  if in_ad_mask.any()  else eval_metrics([], [])
    out_ad  = eval_metrics(y_te[out_ad_mask], p_te[out_ad_mask]) if out_ad_mask.any() else eval_metrics([], [])

    # 6) Coverage + similarity summary
    ad_stats = {
        "ad_threshold": float(ad_threshold),
        "test_in_ad_frac": float(in_ad_mask.mean()),
        "test_in_ad_n": int(in_ad_mask.sum()),
        "test_out_ad_n": int(out_ad_mask.sum()),
        "test_maxsim_mean": float(np.nanmean(te_max_sim)),
        "test_maxsim_median": float(np.nanmedian(te_max_sim)),
        "test_maxsim_p10": float(np.nanpercentile(te_max_sim, 10)),
        "test_maxsim_p90": float(np.nanpercentile(te_max_sim, 90)),
    }

    # 7) Pseudo-neg diagnostics
    pseudo_stats = {
        "n_pseudo": int(len(pseudo)),
        "pseudo_weight": float(pseudo_weight) if pseudo_weight is not None else np.nan,
        "eps": float(eps) if pseudo_weight is not None else np.nan,
        "butina_cutoff_dist": float(butina_cutoff_dist) if pseudo_weight is not None else np.nan,
        "pseudo_ad_threshold": float(pseudo_ad_threshold) if (pseudo_weight is not None and pseudo_ad_threshold is not None) else np.nan,
        "pseudo_maxsim_mean": float(np.nanmean(pseudo_sim)) if (pseudo_sim is not None and len(pseudo_sim) > 0) else np.nan,
        "pseudo_maxsim_median": float(np.nanmedian(pseudo_sim)) if (pseudo_sim is not None and len(pseudo_sim) > 0) else np.nan,
        "pseudo_maxsim_p10": float(np.nanpercentile(pseudo_sim, 10)) if (pseudo_sim is not None and len(pseudo_sim) > 0) else np.nan,
        "pseudo_maxsim_p90": float(np.nanpercentile(pseudo_sim, 90)) if (pseudo_sim is not None and len(pseudo_sim) > 0) else np.nan,
    }

    return overall, in_ad, out_ad, ad_stats, pseudo_stats


# ================================================================
# (8) CV runners
# ================================================================

def run_scaffoldcv(chembl_df, model_name, pseudo_weight=None, eps=0.01, butina_cutoff_dist=0.65,
                   max_coco_candidates=20000, n_splits=10, ad_threshold=0.40, pseudo_ad_threshold=0.40):

    y = chembl_df["y"].values.astype(int)
    scaff = chembl_df["scaffold"].values
    fp_all = chembl_df["fp"].tolist()
    X = fps_to_numpy(fp_all)

    gkf = GroupKFold(n_splits=n_splits)

    rows = []
    ad_rows = []

    for fold, (tr, te) in enumerate(gkf.split(X, y, groups=scaff), start=1):
        X_tr, y_tr = X[tr], y[tr]
        X_te, y_te = X[te], y[te]
        fp_tr = [fp_all[i] for i in tr]
        fp_te = [fp_all[i] for i in te]

        if pseudo_weight is None:
            print(f"[{model_name} Fold {fold}] running...")
        else:
            n_pos = int((y_tr == 1).sum())
            n_neg = int((y_tr == 0).sum())
            n_need = max(0, n_pos - n_neg)
            print(f"[{model_name} Fold {fold}] train pos={n_pos} neg={n_neg} need pseudo_neg={n_need}")

        overall, in_ad, out_ad, ad_stats, pseudo_stats = run_fold_with_ad(
            X_tr=X_tr, y_tr=y_tr, fp_tr=fp_tr,
            X_te=X_te, y_te=y_te, fp_te=fp_te,
            pseudo_weight=pseudo_weight,
            eps=eps,
            butina_cutoff_dist=butina_cutoff_dist,
            max_coco_candidates=max_coco_candidates,
            ad_threshold=ad_threshold,
            pseudo_ad_threshold=pseudo_ad_threshold,
        )

        # One row per fold with overall + stratified metrics
        row = {
            "model": model_name,
            "fold": fold,
            "n_train_pos": int((y_tr == 1).sum()),
            "n_train_neg": int((y_tr == 0).sum()),
            **pseudo_stats,

            # overall
            "roc_auc": overall["roc_auc"],
            "pr_auc": overall["pr_auc"],
            "mcc": overall["mcc"],
            "bal_acc": overall["bal_acc"],

            # in-AD
            "roc_auc_inAD": in_ad["roc_auc"],
            "pr_auc_inAD": in_ad["pr_auc"],
            "mcc_inAD": in_ad["mcc"],
            "bal_acc_inAD": in_ad["bal_acc"],
            "n_inAD": in_ad["n"],

            # out-AD
            "roc_auc_outAD": out_ad["roc_auc"],
            "pr_auc_outAD": out_ad["pr_auc"],
            "mcc_outAD": out_ad["mcc"],
            "bal_acc_outAD": out_ad["bal_acc"],
            "n_outAD": out_ad["n"],

            # AD stats
            **ad_stats,
        }
        rows.append(row)

        print(f"[{model_name} Fold {fold}] ROC={overall['roc_auc']:.3f} PR={overall['pr_auc']:.3f} "
              f"MCC={overall['mcc']:.3f} BalAcc={overall['bal_acc']:.3f} | inAD={ad_stats['test_in_ad_frac']:.2f}")

        # Diagnostics table: keep exactly what you wrote in your CSV header
        ad_rows.append({
            "model": model_name,
            "fold": fold,
            "n_train_pos": int((y_tr == 1).sum()),
            "n_train_neg": int((y_tr == 0).sum()),
            "n_pseudo": pseudo_stats["n_pseudo"],
            "pseudo_weight": pseudo_stats["pseudo_weight"],
            "eps": pseudo_stats["eps"],
            "butina_cutoff_dist": pseudo_stats["butina_cutoff_dist"],
            "pseudo_ad_threshold": pseudo_stats["pseudo_ad_threshold"],
            "pseudo_maxsim_mean": pseudo_stats["pseudo_maxsim_mean"],
            "pseudo_maxsim_median": pseudo_stats["pseudo_maxsim_median"],
            "pseudo_maxsim_p10": pseudo_stats["pseudo_maxsim_p10"],
            "pseudo_maxsim_p90": pseudo_stats["pseudo_maxsim_p90"],

            "roc_auc": overall["roc_auc"],
            "pr_auc":  overall["pr_auc"],
            "mcc":     overall["mcc"],
            "bal_acc": overall["bal_acc"],

            "roc_auc_inAD": in_ad["roc_auc"],
            "pr_auc_inAD":  in_ad["pr_auc"],
            "mcc_inAD":     in_ad["mcc"],
            "bal_acc_inAD": in_ad["bal_acc"],
            "n_inAD":       in_ad["n"],

            "roc_auc_outAD": out_ad["roc_auc"],
            "pr_auc_outAD":  out_ad["pr_auc"],
            "mcc_outAD":     out_ad["mcc"],
            "bal_acc_outAD": out_ad["bal_acc"],
            "n_outAD":       out_ad["n"],

            "ad_threshold": ad_stats["ad_threshold"],
            "test_in_ad_frac": ad_stats["test_in_ad_frac"],
            "test_in_ad_n": ad_stats["test_in_ad_n"],
            "test_out_ad_n": ad_stats["test_out_ad_n"],
            "test_maxsim_mean": ad_stats["test_maxsim_mean"],
            "test_maxsim_median": ad_stats["test_maxsim_median"],
            "test_maxsim_p10": ad_stats["test_maxsim_p10"],
            "test_maxsim_p90": ad_stats["test_maxsim_p90"],
        })

    df = pd.DataFrame(rows)
    df_ad = pd.DataFrame(ad_rows)
    return df, df_ad

def summarize_runs(df_all, metrics):
    group_cols = ["model", "eps", "butina_cutoff_dist", "pseudo_weight", "ad_threshold", "pseudo_ad_threshold"]
    out = (df_all
           .groupby(group_cols, dropna=False)[metrics]
           .agg(["mean", "std"])
           .reset_index())
    out.columns = ["_".join([c for c in col if c]) if isinstance(col, tuple) else col for col in out.columns]
    return out


# ================================================================
# (9) Statistics helpers (paired folds)
# ================================================================

def paired_stats(df_all, model_a, model_b, metric="mcc"):
    A = df_all[df_all["model"] == model_a][["fold", metric]].rename(columns={metric: f"{metric}_A"})
    B = df_all[df_all["model"] == model_b][["fold", metric]].rename(columns={metric: f"{metric}_B"})
    paired = A.merge(B, on="fold", how="inner").sort_values("fold")

    x = paired[f"{metric}_A"].astype(float).values
    y = paired[f"{metric}_B"].astype(float).values

    d = (y - x)
    d = d[np.isfinite(d)]
    n = len(d)

    out = {"metric": metric, "n": int(n)}

    if n < 2:
        out.update({"t_p": np.nan, "t_stat": np.nan, "w_p": np.nan, "w_stat": np.nan, "dz": np.nan,
                    "delta_mean": float(np.nanmean(d)) if n == 1 else np.nan,
                    "delta_ci_low": np.nan, "delta_ci_high": np.nan})
        return out

    t = ttest_rel(x, y, nan_policy="omit")
    out["t_stat"] = float(t.statistic)
    out["t_p"] = float(t.pvalue)

    try:
        w = wilcoxon(x, y)
        out["w_stat"] = float(w.statistic)
        out["w_p"] = float(w.pvalue)
    except Exception:
        out["w_stat"] = np.nan
        out["w_p"] = np.nan

    sd = float(np.std(d, ddof=1))
    out["delta_mean"] = float(np.mean(d))
    out["delta_sd"] = sd
    out["dz"] = float(out["delta_mean"] / sd) if sd > 0 else np.nan

    rng = np.random.default_rng(RANDOM_STATE)
    Bn = 5000
    boots = []
    for _ in range(Bn):
        samp = rng.choice(d, size=n, replace=True)
        boots.append(np.mean(samp))
    boots = np.sort(np.array(boots, dtype=float))
    out["delta_ci_low"] = float(np.percentile(boots, 2.5))
    out["delta_ci_high"] = float(np.percentile(boots, 97.5))

    return out


# ================================================================
# (10) Ablation suite + AD
# ================================================================

def run_ablation_suite_with_ad(chembl_df):
    print("\n=== A) Baseline (ChEMBL only) ===")
    df_base, df_base_ad = run_scaffoldcv(
        chembl_df,
        model_name="baseline",
        pseudo_weight=None,
        n_splits=N_SPLITS,
        ad_threshold=AD_THRESHOLD,
        pseudo_ad_threshold=PSEUDO_AD_THRESHOLD,
    )

    print("\n=== B) Pseudo-neg augmentation (weight=1.0) ===")
    df_w1, df_w1_ad = run_scaffoldcv(
        chembl_df,
        model_name="pseudo_w1",
        pseudo_weight=1.0,
        eps=EPS,
        butina_cutoff_dist=BUTINA_CUTOFF_DIST,
        max_coco_candidates=MAX_COCO_CANDIDATES,
        n_splits=N_SPLITS,
        ad_threshold=AD_THRESHOLD,
        pseudo_ad_threshold=PSEUDO_AD_THRESHOLD,
    )

    print("\n=== C) Pseudo-neg augmentation (weight=0.3) ===")
    df_w03, df_w03_ad = run_scaffoldcv(
        chembl_df,
        model_name="pseudo_w03",
        pseudo_weight=0.3,
        eps=EPS,
        butina_cutoff_dist=BUTINA_CUTOFF_DIST,
        max_coco_candidates=MAX_COCO_CANDIDATES,
        n_splits=N_SPLITS,
        ad_threshold=AD_THRESHOLD,
        pseudo_ad_threshold=PSEUDO_AD_THRESHOLD,
    )

    df_all = pd.concat([df_base, df_w1, df_w03], ignore_index=True)
    df_ad  = pd.concat([df_base_ad, df_w1_ad, df_w03_ad], ignore_index=True)

    metrics = [
        "roc_auc","pr_auc","mcc","bal_acc",
        "roc_auc_inAD","pr_auc_inAD","mcc_inAD","bal_acc_inAD",
        "roc_auc_outAD","pr_auc_outAD","mcc_outAD","bal_acc_outAD",
        "test_in_ad_frac","test_maxsim_mean","test_maxsim_median",
        "n_pseudo","pseudo_maxsim_mean","pseudo_maxsim_median"
    ]
    df_sum = summarize_runs(df_all, metrics=metrics)

    return df_all, df_sum, df_ad


# ================================================================
# (11) UMAP plotting (matplotlib, publication-ready)
# ================================================================

def _require_umap():
    try:
        import umap
        return umap
    except Exception as e:
        raise ImportError(
            "UMAP requires 'umap-learn'. Install via:\n"
            "  pip install umap-learn\n"
            "or\n"
            "  conda install -c conda-forge umap-learn\n"
            f"Original error: {e}"
        )

def _save_pub_figure(fig, out_prefix, dpi=300):
    png = out_prefix + ".png"
    pdf = out_prefix + ".pdf"
    fig.savefig(png, dpi=dpi, bbox_inches="tight")
    fig.savefig(pdf, bbox_inches="tight")
    plt.close(fig)
    return png, pdf

def _umap_fit_transform(X_fit, X_all, n_neighbors=15, min_dist=0.10, random_state=0, metric="jaccard"):
    umap = _require_umap()
    reducer = umap.UMAP(
        n_neighbors=int(n_neighbors),
        min_dist=float(min_dist),
        n_components=2,
        metric=metric,                 # "jaccard" ~ Tanimoto-ish for binary
        random_state=int(random_state),
        transform_seed=int(random_state),
        verbose=False,
    )
    reducer.fit(X_fit)
    Z = reducer.transform(X_all)
    return Z, reducer

def plot_umap_global(
    chembl_df,
    out_prefix,
    n_neighbors=15,
    min_dist=0.10,
    random_state=0,
    point_size=18,
    max_points=4000,
    metric="jaccard"
):
    df = chembl_df.copy()
    if len(df) > int(max_points):
        df = df.sample(n=int(max_points), random_state=int(random_state)).copy()

    X = fps_to_numpy(df["fp"].tolist())
    Z, _ = _umap_fit_transform(X_fit=X, X_all=X, n_neighbors=n_neighbors, min_dist=min_dist,
                              random_state=random_state, metric=metric)

    y = df["y"].astype(int).values

    fig, ax = plt.subplots(figsize=(7.2, 6.2))
    ax.scatter(Z[y == 0, 0], Z[y == 0, 1], s=point_size, alpha=0.65, label="Inactive (0)")
    ax.scatter(Z[y == 1, 0], Z[y == 1, 1], s=point_size, alpha=0.65, label="Active (1)")
    ax.set_xlabel("UMAP-1")
    ax.set_ylabel("UMAP-2")
    ax.set_title("UMAP of ChEMBL M3 (Morgan FP)")
    ax.legend(frameon=False, loc="best")
    ax.grid(False)

    png, pdf = _save_pub_figure(fig, out_prefix)
    print("[SAVED]", png)
    print("[SAVED]", pdf)

def plot_umap_fold_train_test(
    chembl_df,
    out_prefix,
    fold_id=1,
    n_splits=10,
    n_neighbors=15,
    min_dist=0.10,
    random_state=0,
    point_size=18,
    max_points_train=5000,
    max_points_test=2500,
    metric="jaccard"
):
    df = chembl_df.copy()

    y_all = df["y"].values.astype(int)
    scaff = df["scaffold"].values
    fp_all = df["fp"].tolist()
    X_all = fps_to_numpy(fp_all)

    gkf = GroupKFold(n_splits=int(n_splits))
    splits = list(gkf.split(X_all, y_all, groups=scaff))
    if not (1 <= int(fold_id) <= len(splits)):
        raise ValueError(f"fold_id must be in [1, {len(splits)}]. Got {fold_id}")

    tr_idx, te_idx = splits[int(fold_id) - 1]
    tr_idx = np.array(tr_idx, dtype=int)
    te_idx = np.array(te_idx, dtype=int)

    rng = np.random.default_rng(int(random_state))
    if len(tr_idx) > int(max_points_train):
        tr_idx = rng.choice(tr_idx, size=int(max_points_train), replace=False)
    if len(te_idx) > int(max_points_test):
        te_idx = rng.choice(te_idx, size=int(max_points_test), replace=False)

    X_tr = X_all[tr_idx]
    X_te = X_all[te_idx]

    X_concat = np.vstack([X_tr, X_te])
    Z, _ = _umap_fit_transform(X_fit=X_tr, X_all=X_concat, n_neighbors=n_neighbors, min_dist=min_dist,
                              random_state=random_state, metric=metric)

    Z_tr = Z[:len(X_tr)]
    Z_te = Z[len(X_tr):]

    y_tr = y_all[tr_idx]
    y_te = y_all[te_idx]

    fig, ax = plt.subplots(figsize=(7.2, 6.2))

    ax.scatter(Z_tr[y_tr == 0, 0], Z_tr[y_tr == 0, 1], s=point_size, alpha=0.40, marker="o", label="Train inactive")
    ax.scatter(Z_tr[y_tr == 1, 0], Z_tr[y_tr == 1, 1], s=point_size, alpha=0.40, marker="o", label="Train active")

    ax.scatter(Z_te[y_te == 0, 0], Z_te[y_te == 0, 1], s=point_size * 1.1, alpha=0.85, marker="^", label="Test inactive")
    ax.scatter(Z_te[y_te == 1, 0], Z_te[y_te == 1, 1], s=point_size * 1.1, alpha=0.85, marker="^", label="Test active")

    ax.set_xlabel("UMAP-1")
    ax.set_ylabel("UMAP-2")
    ax.set_title(f"UMAP fold {int(fold_id)}: train vs test (fit on train)")
    ax.legend(frameon=False, loc="best", ncols=2)
    ax.grid(False)

    png, pdf = _save_pub_figure(fig, out_prefix)
    print("[SAVED]", png)
    print("[SAVED]", pdf)

def umap_quick_test():
    FIGDIR = os.path.join(BASE, "figures", "UMAP_plots")
    os.makedirs(FIGDIR, exist_ok=True)
    print("[UMAP] FIGDIR:", FIGDIR)

    chembl_df, _ = load_chembl_train(TRAIN_CSV)
    print("[UMAP] Loaded:", chembl_df.shape)

    plot_umap_global(
        chembl_df,
        out_prefix=os.path.join(FIGDIR, "UMAP_global_activity"),
        n_neighbors=15,
        min_dist=0.10,
        random_state=0,
        point_size=18
    )

    plot_umap_fold_train_test(
        chembl_df,
        out_prefix=os.path.join(FIGDIR, "UMAP_fold01_train_vs_test"),
        fold_id=1,
        n_splits=N_SPLITS,
        n_neighbors=15,
        min_dist=0.10,
        random_state=0,
        point_size=18
    )


# ================================================================
# (12) Main
# ================================================================

def main():
    chembl_df, smiles_col = load_chembl_train(TRAIN_CSV)

    print("Loaded:", chembl_df.shape)
    print("Label counts:\n", chembl_df["consensus_label"].value_counts())

    df_all, df_sum, df_ad = run_ablation_suite_with_ad(chembl_df)

    out_all = os.path.join(OUTDIR, "scaffoldcv_ablation_perfold_with_AD.csv")
    out_sum = os.path.join(OUTDIR, "scaffoldcv_ablation_summary_with_AD.csv")
    out_ad  = os.path.join(OUTDIR, "scaffoldcv_AD_diagnostics.csv")

    df_all.to_csv(out_all, index=False)
    df_sum.to_csv(out_sum, index=False)
    df_ad.to_csv(out_ad, index=False)

    print("\nSaved:")
    print(" -", out_all)
    print(" -", out_sum)
    print(" -", out_ad)

    print("\nAblation summary (mean±sd):")
    print(df_sum)

    print("\n=== Paired fold statistics: baseline vs pseudo_w1 ===")
    for metric in ["mcc", "bal_acc", "roc_auc", "mcc_inAD", "bal_acc_inAD"]:
        st = paired_stats(df_all, model_a="baseline", model_b="pseudo_w1", metric=metric)
        print(f"{metric}: Δmean={st['delta_mean']:+.4f} "
              f"(95% CI {st['delta_ci_low']:+.4f} to {st['delta_ci_high']:+.4f}) | "
              f"t p={st['t_p']:.4g} | Wilcoxon p={st['w_p']:.4g} | dz={st['dz']:.3f}")

    cov = (df_all.groupby("model")["test_in_ad_frac"]
           .agg(["mean","std","min","max"])
           .reset_index())
    print("\nAD coverage (test_in_ad_frac):")
    print(cov)

    return df_all, df_sum, df_ad


# ================================================================
# Notebook run options
# ================================================================

# 1) FAST: only generate UMAP figures (no COCONUT, no CV):
umap_quick_test()

# 2) FULL: run the whole pipeline (CV + COCONUT) and write CSVs:
df_all, df_sum, df_ad = main()


[UMAP] FIGDIR: C:\Users\Besitzer\Desktop\M3_databases\figures\UMAP_plots
[UMAP] Loaded: (2268, 17)


c:\Users\Besitzer\anaconda3\envs\openms_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Besitzer\anaconda3\envs\openms_env\lib\site-packages\umap\umap_.py:1887: UserWarning: gradient function is not yet implemented for jaccard distance metric; inverse_transform will be unavailable
  warn(
c:\Users\Besitzer\anaconda3\envs\openms_env\lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


[SAVED] C:\Users\Besitzer\Desktop\M3_databases\figures\UMAP_plots\UMAP_global_activity.png
[SAVED] C:\Users\Besitzer\Desktop\M3_databases\figures\UMAP_plots\UMAP_global_activity.pdf


c:\Users\Besitzer\anaconda3\envs\openms_env\lib\site-packages\umap\umap_.py:1887: UserWarning: gradient function is not yet implemented for jaccard distance metric; inverse_transform will be unavailable
  warn(
c:\Users\Besitzer\anaconda3\envs\openms_env\lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


[SAVED] C:\Users\Besitzer\Desktop\M3_databases\figures\UMAP_plots\UMAP_fold01_train_vs_test.png
[SAVED] C:\Users\Besitzer\Desktop\M3_databases\figures\UMAP_plots\UMAP_fold01_train_vs_test.pdf
Loaded: (2268, 17)
Label counts:
 consensus_label
active_single      1502
inactive_single     463
active              286
inactive             17
Name: count, dtype: int64

=== A) Baseline (ChEMBL only) ===
[baseline Fold 1] running...


...

[pseudo_w1 Fold 3] ROC=0.996 PR=0.999 MCC=0.875 BalAcc=0.924 | inAD=0.98
[pseudo_w1 Fold 4] train pos=1593 neg=448 need pseudo_neg=1145


[21:45:05] Can't kekulize mol.  Unkekulized atoms: 1 2 3 4 5 6 10 11 15 16 17 19 20 21
[21:45:05] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19


[pseudo_w1 Fold 4] ROC=0.981 PR=0.997 MCC=0.766 BalAcc=0.826 | inAD=0.95
[pseudo_w1 Fold 5] train pos=1596 neg=445 need pseudo_neg=1151


[21:47:16] Can't kekulize mol.  Unkekulized atoms: 1 2 3 4 5 6 10 11 15 16 17 19 20 21
[21:47:16] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19


KeyboardInterrupt: 

In [2]:
import os
os.system("shutdown /h")

0